In [ ]:
# In[1]:

import pandas as pd
import pytz
import datetime as dt

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are in the current working directory)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Ensure timestamp is integer where present
for df in (metric_df, trace_df, log_df, error_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = df['timestamp'].astype('Int64')

# helper to convert epoch seconds to Asia/Shanghai string
def ts_to_shanghai_str(ts):
    if pd.isna(ts):
        return None
    # ts may be numpy int; convert to int
    ts_i = int(ts)
    return dt.datetime.fromtimestamp(ts_i, tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# -------------------------
# metric.csv summaries
# -------------------------
metric_total = metric_df.shape[0]
metric_min_ts = metric_df['timestamp'].min() if metric_total>0 else pd.NA
metric_max_ts = metric_df['timestamp'].max() if metric_total>0 else pd.NA
metric_min_ts_str = ts_to_shanghai_str(metric_min_ts)
metric_max_ts_str = ts_to_shanghai_str(metric_max_ts)

metric_cmdb_counts = metric_df['cmdb_id'].value_counts().reset_index()
metric_cmdb_counts.columns = ['cmdb_id','count']
metric_cmdb_counts = metric_cmdb_counts.head(50)

metric_kpi_counts = metric_df['kpi_name'].value_counts().reset_index()
metric_kpi_counts.columns = ['kpi_name','count']
metric_kpi_counts = metric_kpi_counts.head(50)

metric_summary_df = pd.DataFrame({
    'file': ['metric.csv'],
    'total_rows': [metric_total],
    'min_timestamp_Asia/Shanghai': [metric_min_ts_str],
    'max_timestamp_Asia/Shanghai': [metric_max_ts_str]
})

# -------------------------
# trace.csv summaries
# -------------------------
trace_total = trace_df.shape[0]
trace_min_ts = trace_df['timestamp'].min() if trace_total>0 else pd.NA
trace_max_ts = trace_df['timestamp'].max() if trace_total>0 else pd.NA
trace_min_ts_str = ts_to_shanghai_str(trace_min_ts)
trace_max_ts_str = ts_to_shanghai_str(trace_max_ts)

trace_cmdb_counts = trace_df['cmdb_id'].value_counts().reset_index()
trace_cmdb_counts.columns = ['cmdb_id','count']
trace_cmdb_counts = trace_cmdb_counts.head(50)

trace_name_counts = trace_df['trace_name'].value_counts().reset_index()
trace_name_counts.columns = ['trace_name','count']
trace_name_counts = trace_name_counts.head(50)

trace_summary_df = pd.DataFrame({
    'file': ['trace.csv'],
    'total_rows': [trace_total],
    'min_timestamp_Asia/Shanghai': [trace_min_ts_str],
    'max_timestamp_Asia/Shanghai': [trace_max_ts_str]
})

# -------------------------
# log.csv summaries
# -------------------------
log_total = log_df.shape[0]
log_min_ts = log_df['timestamp'].min() if log_total>0 else pd.NA
log_max_ts = log_df['timestamp'].max() if log_total>0 else pd.NA
log_min_ts_str = ts_to_shanghai_str(log_min_ts)
log_max_ts_str = ts_to_shanghai_str(log_max_ts)

log_cmdb_counts = log_df['cmdb_id'].value_counts().reset_index()
log_cmdb_counts.columns = ['cmdb_id','count']
log_cmdb_counts = log_cmdb_counts.head(50)

log_name_counts = log_df['log_name'].value_counts().reset_index()
log_name_counts.columns = ['log_name','count']
log_name_counts = log_name_counts.head(50)

log_summary_df = pd.DataFrame({
    'file': ['log.csv'],
    'total_rows': [log_total],
    'min_timestamp_Asia/Shanghai': [log_min_ts_str],
    'max_timestamp_Asia/Shanghai': [log_max_ts_str]
})

# -------------------------
# error_logs.csv summaries
# -------------------------
error_total = error_df.shape[0]
error_min_ts = error_df['timestamp'].min() if error_total>0 else pd.NA
error_max_ts = error_df['timestamp'].max() if error_total>0 else pd.NA
error_min_ts_str = ts_to_shanghai_str(error_min_ts)
error_max_ts_str = ts_to_shanghai_str(error_max_ts)

error_cmdb_counts = error_df['cmdb_id'].value_counts().reset_index()
error_cmdb_counts.columns = ['cmdb_id','count']
error_cmdb_counts = error_cmdb_counts.head(50)

# Prepare earliest 5 and latest 5 records with converted timestamp strings
if error_total>0:
    error_sorted = error_df.sort_values('timestamp', ascending=True)
    error_earliest5 = error_sorted.head(5).copy()
    error_latest5 = error_sorted.tail(5).copy()
    # Add readable timestamp column in Asia/Shanghai
    error_earliest5['timestamp_shanghai'] = error_earliest5['timestamp'].apply(ts_to_shanghai_str)
    error_latest5['timestamp_shanghai'] = error_latest5['timestamp'].apply(ts_to_shanghai_str)
    # select only requested columns
    error_earliest5 = error_earliest5[['timestamp_shanghai','timestamp','cmdb_id','message']].rename(columns={'timestamp_shanghai':'ts_Asia/Shanghai'})
    error_latest5 = error_latest5[['timestamp_shanghai','timestamp','cmdb_id','message']].rename(columns={'timestamp_shanghai':'ts_Asia/Shanghai'})
else:
    error_earliest5 = pd.DataFrame(columns=['ts_Asia/Shanghai','timestamp','cmdb_id','message'])
    error_latest5 = pd.DataFrame(columns=['ts_Asia/Shanghai','timestamp','cmdb_id','message'])

error_summary_df = pd.DataFrame({
    'file': ['error_logs.csv'],
    'total_rows': [error_total],
    'min_timestamp_Asia/Shanghai': [error_min_ts_str],
    'max_timestamp_Asia/Shanghai': [error_max_ts_str]
})

# Final compact outputs (use variables rather than print)
# For each file: summary row, cmdb counts (top 50), name/counts (top50 where applicable)
metric_summary_df, metric_cmdb_counts, metric_kpi_counts, trace_summary_df, trace_cmdb_counts, trace_name_counts, log_summary_df, log_cmdb_counts, log_name_counts, error_summary_df, error_cmdb_counts, error_earliest5, error_latest5

```
Out[1]:
```
```python
# Reuse existing variables from prior execution to build a concise plain-English summary string.
# Final output is a single compact string variable 'summary'.

top_metric_cmdb = metric_cmdb_counts.head(5).to_records(index=False)
top_metric_kpis = metric_kpi_counts.head(3).to_records(index=False)

top_trace_cmdb = trace_cmdb_counts.head(5).to_records(index=False)
top_trace_names = trace_name_counts.head(3).to_records(index=False)

top_log_cmdb = log_cmdb_counts.head(5).to_records(index=False)
top_log_names = log_name_counts.head(3).to_records(index=False)

top_error_cmdb = error_cmdb_counts.head(5).to_records(index=False)

summary = (
    f"Summary of telemetry files (timestamps shown in Asia/Shanghai):\n\n"
    # metric
    f"metric.csv: {metric_total} rows, time range {metric_min_ts_str} to {metric_max_ts_str}.\n"
    f" Top cmdb_ids (top 5): " +
    ", ".join([f\"{r[0]}({r[1]})\" for r in top_metric_cmdb]) + ".\n"
    f" Top KPI names (sample 3): " +
    ", ".join([f\"{r[0]}({r[1]})\" for r in top_metric_kpis]) + ".\n\n"
    # trace
    f"trace.csv: {trace_total} rows, time range {trace_min_ts_str} to {trace_max_ts_str}.\n"
    f" Top cmdb_ids (top 5): " +
    ", ".join([f\"{r[0]}({r[1]})\" for r in top_trace_cmdb]) + ".\n"
    f" Top trace names (sample 3): " +
    ", ".join([f\"{r[0]}({r[1]})\" for r in top_trace_names]) + ".\n\n"
    # log
    f"log.csv: {log_total} rows, time range {log_min_ts_str} to {log_max_ts_str}.\n"
    f" Top cmdb_ids (top 5): " +
    ", ".join([f\"{r[0]}({r[1]})\" for r in top_log_cmdb]) + ".\n"
    f" Log names: " + ", ".join([f\"{r[0]}({r[1]})\" for r in top_log_names]) + ".\n\n"
    # error_logs
    f"error_logs.csv: {error_total} rows, time range {error_min_ts_str} to {error_max_ts_str}.\n"
    f" Top cmdb_ids (top 5): " +
    ", ".join([f\"{r[0]}({r[1]})\" for r in top_error_cmdb]) + ".\n"
    f" Earliest 5 and latest 5 error-log samples were extracted (see prior output)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         file  total_rows min_timestamp_Asia/Shanghai max_timestamp_Asia/Shanghai
0  metric.csv      151992     2022-03-21 19:30:00 CST     2022-03-21 19:59:00 CST,           cmdb_id  count
0       adservice  10230
1      adservice2   9990
2     frontend2-0   5520
3      frontend-2   5400
4      frontend-1   5400
..            ...    ...
45         node-1   1692
46         node-3   1692
47         node-5   1692
48  redis-cart2-0    360
49   redis-cart-0    360

[50 rows x 2 columns],                                              kpi_name  count
0   container.node-5.container_network_transmit_pa...   1050
1   container.node-5.container_network_transmit_MB...   1050
2   container.node-5.container_network_transmit_er...   1050
3   container.node-5.container_network_receive_pac...   1050
4   container.node-5.container_network_receive_err...   1050
..                                                ...    ...
45      container.node-5.container_start_time_seconds    990
46     container.node-5.container_tasks_state.running    990
47                 container.node-5.container_threads    990
48  container.node-5.container_tasks_state.uninter...    990
49  container.node-5.container_fs_reads_merged./de...    990

[50 rows x 2 columns],         file  total_rows min_timestamp_Asia/Shanghai max_timestamp_Asia/Shanghai
0  trace.csv       36784     2022-03-21 19:30:00 CST     2022-03-21 19:59:00 CST,                      cmdb_id  count
0                 frontend-0   2760
1                 frontend-2   2752
2          checkoutservice-0   2744
3          checkoutservice-2   2732
4                 frontend-1   2716
..                       ...    ...
35        currencyservice2-0    240
36  productcatalogservice2-0    240
37              adservice2-0    120
38           emailservice2-0    120
39         paymentservice2-0    120

[40 rows x 2 columns],                               trace_name  count
0    trace.from_frontend-0.duration_mean    630
1     trace.from_frontend-0.duration_p95    630
2       trace.from_frontend-0.error_rate    630
3        trace.from_frontend-0.row_count    630
4        trace.from_frontend-2.row_count    628
..                                   ...    ...
45   trace.from_frontend2-0.duration_p95    210
46     trace.from_frontend2-0.error_rate    210
47      trace.from_frontend2-0.row_count    210
48  trace.to_cartservice-0.duration_mean    180
49   trace.to_cartservice-0.duration_p95    180

[50 rows x 2 columns],       file  total_rows min_timestamp_Asia/Shanghai max_timestamp_Asia/Shanghai
0  log.csv        1814     2022-03-21 19:30:00 CST     2022-03-21 19:59:00 CST,                     cmdb_id  count
0               adservice-0     60
1               adservice-1     60
2               adservice-2     60
3             cartservice-0     60
4             cartservice-1     60
..                      ...    ...
26  recommendationservice-2     60
27        shippingservice-0     60
28        shippingservice-1     60
29        shippingservice-2     60
30             redis-cart-0     14

[31 rows x 2 columns],           log_name  count
0  log.error_count    907
1    log.row_count    907,              file  total_rows min_timestamp_Asia/Shanghai max_timestamp_Asia/Shanghai
0  error_logs.csv        1929     2022-03-21 19:30:01 CST     2022-03-21 19:59:59 CST,       cmdb_id  count
0  frontend-0    922
1  frontend-2    543
2  frontend-1    464,           ts_Asia/Shanghai   timestamp     cmdb_id                                            message
0  2022-03-21 19:30:01 CST  1647862201  frontend-2  severity: warning, message: failed to retrieve...
1  2022-03-21 19:30:01 CST  1647862201  frontend-0  severity: warning, message: failed to retrieve...
2  2022-03-21 19:30:02 CST  1647862202  frontend-1  severity: warning, message: failed to retrieve...
3  2022-03-21 19:30:03 CST  1647862203  frontend-0  severity: warning, message: failed to retrieve...
4  2022-03-21 19:30:04 CST  1647862204  frontend-2  severity: warning, message: failed to retrieve...,              ts_Asia/Shanghai   timestamp     cmdb_id                                            message
1924  2022-03-21 19:59:54 CST  1647863994  frontend-2  severity: warning, message: failed to retrieve...
1926  2022-03-21 19:59:56 CST  1647863996  frontend-0  severity: warning, message: failed to retrieve...
1925  2022-03-21 19:59:56 CST  1647863996  frontend-1  severity: warning, message: failed to retrieve...
1927  2022-03-21 19:59:58 CST  1647863998  frontend-0  severity: warning, message: failed to retrieve...
1928  2022-03-21 19:59:59 CST  1647863999  frontend-0  severity: warning, message: failed to retrieve...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Use existing metric_df, ts_to_shanghai_str from prior state.
# Compute global P95 and P5 per (cmdb_id, kpi_name) across full series
thresholds = (
    metric_df
    .groupby(['cmdb_id','kpi_name'], observed=True)['value']
    .agg(P95=lambda s: s.quantile(0.95), P5=lambda s: s.quantile(0.05))
    .reset_index()
)

# Filter metric points inside the incident window (inclusive)
start_ts, end_ts = 1647862200, 1647863999
window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# Merge thresholds to window points so we can evaluate anomalies with group-specific thresholds
merged = window_df.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

# Mark high and low anomalies (>=P95, <=P5). NaNs in thresholds lead to False.
merged['high'] = merged['value'] >= merged['P95']
merged['low'] = merged['value'] <= merged['P5']

# Aggregate counts per pair within the window
group_window_counts = (
    merged
    .groupby(['cmdb_id','kpi_name'], observed=True)
    .agg(
        count_in_window=('value','count'),
        count_high=('high','sum'),
        count_low=('low','sum')
    )
    .reset_index()
)

# Earliest anomaly timestamp per pair (consider both high and low anomalies)
anomalies = merged[merged['high'] | merged['low']].copy()
earliest = (
    anomalies
    .groupby(['cmdb_id','kpi_name'], observed=True)['timestamp']
    .min()
    .reset_index()
    .rename(columns={'timestamp':'earliest_anomaly_ts'})
)

# Combine thresholds, window counts, and earliest anomaly timestamp
result = (
    thresholds
    .merge(group_window_counts, on=['cmdb_id','kpi_name'], how='left')
    .merge(earliest, on=['cmdb_id','kpi_name'], how='left')
)

# Fill NaN counts with 0 and ensure integer dtype where appropriate
for c in ['count_in_window','count_high','count_low']:
    if c in result.columns:
        result[c] = result[c].fillna(0).astype(int)

# Compute total anomalies and keep only pairs with any anomalies in the window
result['total_anomalies'] = result['count_high'] + result['count_low']
anomalous_pairs = result[result['total_anomalies'] > 0].copy()

# Convert earliest anomaly timestamp to Asia/Shanghai string for readability
if not anomalous_pairs.empty:
    anomalous_pairs['earliest_anomaly_ts_shanghai'] = anomalous_pairs['earliest_anomaly_ts'].apply(lambda x: ts_to_shanghai_str(x) if pd.notna(x) else pd.NA)

# Select and order columns, sort by total_anomalies desc, limit to top 20
cols = ['cmdb_id','kpi_name','P95','P5','count_in_window','count_high','count_low','total_anomalies','earliest_anomaly_ts','earliest_anomaly_ts_shanghai']
final_result = anomalous_pairs.sort_values('total_anomalies', ascending=False).loc[:, cols].head(20)

# If no anomalies found, return a concise message instead of DataFrame
if final_result.shape[0] == 0:
    final_result = "No anomalies (>=P95 or <=P5) were found for any (cmdb_id, kpi_name) within the incident window."

final_result

```
Out[2]:
```
Summary (plain English):

- Top anomalous (cmdb_id, kpi_name) pairs (limit 20): all 20 pairs are for cmdb_id = shippingservice2-0 and are various container.node-5.* metrics (filesystems, I/O, memory failcnt, threads, file descriptors, etc.).

- For each of these 20 pairs:
  - Global P95 and P5 (computed over the full series) are shown per KPI. Many KPIs have identical P95 and P5 (e.g., 0.0 or fixed constants).
  - Points inside the incident window [1647862200, 1647863999] (2022-03-21 19:30:00 to 19:59:59 Asia/Shanghai): count_in_window = 30.
  - Count of high-side anomalies (value >= P95): 30.
  - Count of low-side anomalies (value <= P5): 30.
  - total_anomalies reported = 60 (this is the sum of high + low; note this double-counts points when P95 == P5 and values equal the threshold).
  - Earliest anomaly timestamp in the window = 1647862200 (2022-03-21 19:30:00 Asia/Shanghai).

- Interpretation / note:
  - The anomaly counts reflect that for these KPIs the global P95 equals the global P5 (or that observed values equal those thresholds), so every in-window point meets both >=P95 and <=P5. That causes each point to be counted as both a high and a low anomaly (hence count_high=30, count_low=30, total_anomalies=60). This is a threshold-artifact rather than two distinct opposite-sign anomalies.
  - All top anomalous signals are concentrated on shippingservice2-0's container.node-5 related metrics and began at the start of the incident window (19:30:00 CST).

If you want, I can:
- Deduplicate anomalies so each point is counted once (high-preference or low-preference) and re-rank,
- Or show the full top-20 table again with selected columns.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name           P95            P5  count_in_window  count_high  count_low  total_anomalies  earliest_anomaly_ts earliest_anomaly_ts_shanghai
5059  shippingservice2-0  container.node-5.container_ulimits_soft.max_op...  6.553600e+04  6.553600e+04               30          30         30               60           1647862200      2022-03-21 19:30:00 CST
5058  shippingservice2-0             container.node-5.container_threads_max  0.000000e+00  0.000000e+00               30          30         30               60           1647862200      2022-03-21 19:30:00 CST
5018  shippingservice2-0  container.node-5.container_fs_write_seconds./d...  0.000000e+00  0.000000e+00               30          30         30               60           1647862200      2022-03-21 19:30:00 CST
5019  shippingservice2-0      container.node-5.container_fs_writes./dev/vda  0.000000e+00  0.000000e+00               30          30         30               60           1647862200      2022-03-21 19:30:00 CST
5020  shippingservice2-0     container.node-5.container_fs_writes./dev/vda1  0.000000e+00  0.000000e+00               30          30         30               60           1647862200      2022-03-21 19:30:00 CST
5021  shippingservice2-0   container.node-5.container_fs_writes_MB./dev/vda  0.000000e+00  0.000000e+00               30          30         30               60           1647862200      2022-03-21 19:30:00 CST
5022  shippingservice2-0  container.node-5.container_fs_writes_merged./d...  0.000000e+00  0.000000e+00               30          30         30               60           1647862200      2022-03-21 19:30:00 CST
5025  shippingservice2-0          container.node-5.container_memory_failcnt  2.399892e+06  2.399892e+06               30          30         30               60           1647862200      2022-03-21 19:30:00 CST
5010  shippingservice2-0  container.node-5.container_fs_read_seconds./de...  0.000000e+00  0.000000e+00               30          30         30               60           1647862200      2022-03-21 19:30:00 CST
5012  shippingservice2-0      container.node-5.container_fs_reads./dev/vda1  0.000000e+00  0.000000e+00               30          30         30               60           1647862200      2022-03-21 19:30:00 CST
5014  shippingservice2-0  container.node-5.container_fs_reads_merged./de...  0.000000e+00  0.000000e+00               30          30         30               60           1647862200      2022-03-21 19:30:00 CST
5015  shippingservice2-0  container.node-5.container_fs_sector_reads./de...  0.000000e+00  0.000000e+00               30          30         30               60           1647862200      2022-03-21 19:30:00 CST
5016  shippingservice2-0  container.node-5.container_fs_sector_writes./d...  0.000000e+00  0.000000e+00               30          30         30               60           1647862200      2022-03-21 19:30:00 CST
5003  shippingservice2-0        container.node-5.container_file_descriptors  1.000000e+01  1.000000e+01               30          30         30               60           1647862200      2022-03-21 19:30:00 CST
5004  shippingservice2-0     container.node-5.container_fs_inodes./dev/vda1  0.000000e+00  0.000000e+00               30          30         30               60           1647862200      2022-03-21 19:30:00 CST
5005  shippingservice2-0  container.node-5.container_fs_inodes_free./dev...  0.000000e+00  0.000000e+00               30          30         30               60           1647862200      2022-03-21 19:30:00 CST
5006  shippingservice2-0  container.node-5.container_fs_io_current./dev/...  0.000000e+00  0.000000e+00               30          30         30               60           1647862200      2022-03-21 19:30:00 CST
5007  shippingservice2-0  container.node-5.container_fs_io_time_seconds....  0.000000e+00  0.000000e+00               30          30         30               60           1647862200      2022-03-21 19:30:00 CST
5008  shippingservice2-0  container.node-5.container_fs_io_time_weighted...  0.000000e+00  0.000000e+00               30          30         30               60           1647862200      2022-03-21 19:30:00 CST
5009  shippingservice2-0   container.node-5.container_fs_limit_MB./dev/vda1  6.046307e+05  6.046307e+05               30          30         30               60           1647862200      2022-03-21 19:30:00 CST```
```

In [ ]:
# In[3]:

# Re-run adjusted analysis for shippingservice2-0 KPIs with robust NA handling.
import pandas as pd
import numpy as np

# Filter for shippingservice2-0 (reuse metric_df)
ship = metric_df[metric_df['cmdb_id'] == 'shippingservice2-0'].copy()

# Global thresholds per kpi_name
thresholds_ship = (
    ship
    .groupby('kpi_name', observed=True)['value']
    .agg(P95=lambda s: s.quantile(0.95), P5=lambda s: s.quantile(0.05))
    .reset_index()
)

# Incident window
start_ts, end_ts = 1647862200, 1647863999
window_ship = ship[(ship['timestamp'] >= start_ts) & (ship['timestamp'] <= end_ts)].copy()

# Merge thresholds into window data
window_merged = window_ship.merge(thresholds_ship, on='kpi_name', how='left')

# Mark anomaly per point (value >= P95 or value <= P5)
window_merged['is_anomaly'] = (window_merged['value'] >= window_merged['P95']) | (window_merged['value'] <= window_merged['P5'])

# Helper to compute longest consecutive run (timestamps in seconds, step=60)
def longest_run_info(timestamps):
    if len(timestamps) == 0:
        return 0, pd.NA
    ts = np.array(sorted(pd.Series(timestamps).dropna().astype(int).unique()))
    if ts.size == 0:
        return 0, pd.NA
    diffs = np.diff(ts)
    runs = []
    run_start = int(ts[0])
    run_len = 1
    for i, d in enumerate(diffs):
        if d == 60:
            run_len += 1
        else:
            runs.append((run_len, run_start))
            run_start = int(ts[i+1])
            run_len = 1
    runs.append((run_len, run_start))
    runs_sorted = sorted(runs, key=lambda x: (-x[0], x[1]))
    best_len, best_start = runs_sorted[0]
    return int(best_len), int(best_start)

# Aggregate per kpi_name
rows = []
for kpi, g in window_merged.groupby('kpi_name', observed=True):
    count_points = int(g.shape[0])
    anomalous_ts = g.loc[g['is_anomaly'], 'timestamp'].dropna().astype(int).unique().tolist()
    unique_anomaly_count = int(len(anomalous_ts))
    anomaly_fraction = float(unique_anomaly_count / count_points) if count_points > 0 else 0.0
    earliest_anom_ts = int(min(anomalous_ts)) if unique_anomaly_count > 0 else pd.NA
    longest_len, longest_start_ts = longest_run_info(anomalous_ts)
    rows.append({
        'kpi_name': kpi,
        'count_points_in_window': count_points,
        'unique_anomaly_count': unique_anomaly_count,
        'anomaly_fraction': round(anomaly_fraction, 4),
        'longest_run_length': longest_len,
        'longest_run_start_ts': longest_start_ts,
        'earliest_anomaly_ts': earliest_anom_ts
    })

agg_df = pd.DataFrame(rows)

# Merge with thresholds to include global P95/P5 (keep all KPIs for this cmdb_id)
result = thresholds_ship.merge(agg_df, on='kpi_name', how='left')

# Fill NaNs for KPIs that had no points in window
result['count_points_in_window'] = result['count_points_in_window'].fillna(0).astype(int)
result['unique_anomaly_count'] = result['unique_anomaly_count'].fillna(0).astype(int)
result['anomaly_fraction'] = result['anomaly_fraction'].fillna(0.0).astype(float)
result['longest_run_length'] = result['longest_run_length'].fillna(0).astype(int)

# Convert timestamps to Asia/Shanghai strings when present
def to_shanghai_or_na(x):
    return ts_to_shanghai_str(int(x)) if pd.notna(x) else pd.NA

result['longest_run_start_ts_sh'] = result['longest_run_start_ts'].apply(lambda x: to_shanghai_or_na(x))
result['earliest_anomaly_ts_sh'] = result['earliest_anomaly_ts'].apply(lambda x: to_shanghai_or_na(x))

# Prepare final compact table with requested columns and sort by unique_anomaly_count desc
final = result.rename(columns={'P95':'global_P95','P5':'global_P5'}).loc[:, [
    'kpi_name','global_P95','global_P5',
    'count_points_in_window','unique_anomaly_count','anomaly_fraction',
    'longest_run_length','longest_run_start_ts_sh','earliest_anomaly_ts_sh'
]]

final = final.sort_values('unique_anomaly_count', ascending=False).head(20).reset_index(drop=True)

final

```
Out[3]:
```
Summary (plain English):

- Scope: analysis of KPIs for cmdb_id = shippingservice2-0 within incident window 1647862200–1647863999 (2022-03-21 19:30:00–19:59:59 Asia/Shanghai).

- Main finding: every inspected KPI had all points in the window flagged anomalous.
  - For each KPI shown:
    - count_points_in_window = 30 (one minute samples over 30 minutes).
    - unique_anomaly_count = 30 (every timestamp in the window flagged).
    - anomaly_fraction = 1.0 (100% of points anomalous).
    - longest consecutive anomalous run = 30 minutes, starting at 2022-03-21 19:30:00 Asia/Shanghai.
    - earliest anomaly timestamp = 2022-03-21 19:30:00 Asia/Shanghai.

- Thresholds: global P95 and P5 were computed per KPI from the full series. Many KPIs have P95 == P5 (e.g., several KPIs show both thresholds equal to 0.0, 10.0, 65536, etc.), which explains why every in-window value met the anomaly condition (value >= P95 OR value <= P5).

- Interpretation / caveat: this is likely a threshold-artifact (P95 equals P5) rather than distinct high vs low deviations. All top anomalous signals are concentrated on shippingservice2-0 container.node-5 metrics (filesystems, I/O, threads, file descriptors, cpu/throttle, etc.), and the anomaly run starts at the incident window start (19:30:00).

If you want, I can:
- Deduplicate or change the anomaly rule (e.g., treat equality ties differently),
- Recompute using a different percentile (e.g., P97.5/P2.5) or require strict >P95 / <P5,
- Or show the explicit top-20 KPI table again.

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name     global_P95      global_P5  count_points_in_window  unique_anomaly_count  anomaly_fraction  longest_run_length  longest_run_start_ts_sh   earliest_anomaly_ts_sh
0   container.node-5.container_cpu_cfs_throttled_p...       0.000000       0.000000                      30                    30               1.0                  30  2022-03-21 19:30:00 CST  2022-03-21 19:30:00 CST
1   container.node-5.container_cpu_cfs_throttled_s...       0.000000       0.000000                      30                    30               1.0                  30  2022-03-21 19:30:00 CST  2022-03-21 19:30:00 CST
2     container.node-5.container_cpu_load_average_10s       0.000000       0.000000                      30                    30               1.0                  30  2022-03-21 19:30:00 CST  2022-03-21 19:30:00 CST
3         container.node-5.container_file_descriptors      10.000000      10.000000                      30                    30               1.0                  30  2022-03-21 19:30:00 CST  2022-03-21 19:30:00 CST
4   container.node-5.container_fs_io_time_seconds....       0.000000       0.000000                      30                    30               1.0                  30  2022-03-21 19:30:00 CST  2022-03-21 19:30:00 CST
5   container.node-5.container_fs_io_current./dev/...       0.000000       0.000000                      30                    30               1.0                  30  2022-03-21 19:30:00 CST  2022-03-21 19:30:00 CST
6   container.node-5.container_fs_inodes_free./dev...       0.000000       0.000000                      30                    30               1.0                  30  2022-03-21 19:30:00 CST  2022-03-21 19:30:00 CST
7      container.node-5.container_fs_inodes./dev/vda1       0.000000       0.000000                      30                    30               1.0                  30  2022-03-21 19:30:00 CST  2022-03-21 19:30:00 CST
8   container.node-5.container_fs_io_time_weighted...       0.000000       0.000000                      30                    30               1.0                  30  2022-03-21 19:30:00 CST  2022-03-21 19:30:00 CST
9    container.node-5.container_fs_limit_MB./dev/vda1  604630.738281  604630.738281                      30                    30               1.0                  30  2022-03-21 19:30:00 CST  2022-03-21 19:30:00 CST
10  container.node-5.container_fs_read_seconds./de...       0.000000       0.000000                      30                    30               1.0                  30  2022-03-21 19:30:00 CST  2022-03-21 19:30:00 CST
11       container.node-5.container_fs_reads./dev/vda       0.550000       0.000000                      30                    30               1.0                  30  2022-03-21 19:30:00 CST  2022-03-21 19:30:00 CST
12  container.node-5.container_fs_sector_reads./de...       0.000000       0.000000                      30                    30               1.0                  30  2022-03-21 19:30:00 CST  2022-03-21 19:30:00 CST
13  container.node-5.container_fs_reads_merged./de...       0.000000       0.000000                      30                    30               1.0                  30  2022-03-21 19:30:00 CST  2022-03-21 19:30:00 CST
14    container.node-5.container_fs_reads_MB./dev/vda       0.005371       0.000000                      30                    30               1.0                  30  2022-03-21 19:30:00 CST  2022-03-21 19:30:00 CST
15      container.node-5.container_fs_reads./dev/vda1       0.000000       0.000000                      30                    30               1.0                  30  2022-03-21 19:30:00 CST  2022-03-21 19:30:00 CST
16  container.node-5.container_tasks_state.uninter...       0.000000       0.000000                      30                    30               1.0                  30  2022-03-21 19:30:00 CST  2022-03-21 19:30:00 CST
17                 container.node-5.container_threads      20.000000      20.000000                      30                    30               1.0                  30  2022-03-21 19:30:00 CST  2022-03-21 19:30:00 CST
18             container.node-5.container_threads_max       0.000000       0.000000                      30                    30               1.0                  30  2022-03-21 19:30:00 CST  2022-03-21 19:30:00 CST
19  container.node-5.container_ulimits_soft.max_op...   65536.000000   65536.000000                      30                    30               1.0                  30  2022-03-21 19:30:00 CST  2022-03-21 19:30:00 CST```
```

In [ ]:
# In[4]:

# Reuse existing trace_df and helper ts_to_shanghai_str from prior state.
import pandas as pd
import numpy as np

cmdb_target = 'shippingservice2-0'
start_ts, end_ts = 1647862200, 1647863999

# Filter trace rows for the target cmdb_id (full series for threshold computation)
trace_ship_full = trace_df[trace_df['cmdb_id'] == cmdb_target].copy()

if trace_ship_full.shape[0] == 0:
    trace_result = f"No trace rows found for cmdb_id == '{cmdb_target}'."
else:
    # Global thresholds (P95, P5) per trace_name using the full series for this cmdb_id
    thresholds_trace = (
        trace_ship_full
        .groupby('trace_name', observed=True)['value']
        .agg(global_P95=lambda s: s.quantile(0.95), global_P5=lambda s: s.quantile(0.05))
        .reset_index()
    )

    # Incident window subset (only rows within the window)
    trace_window = trace_ship_full[(trace_ship_full['timestamp'] >= start_ts) & (trace_ship_full['timestamp'] <= end_ts)].copy()

    # If no points in window, return empty summary
    if trace_window.shape[0] == 0:
        trace_result = f"No trace rows for {cmdb_target} within the incident window {start_ts}-{end_ts}."
    else:
        # Merge thresholds into window rows
        merged = trace_window.merge(thresholds_trace, on='trace_name', how='left')

        # Mark anomaly per row (>=P95 or <=P5)
        merged['is_anomaly'] = (merged['value'] >= merged['global_P95']) | (merged['value'] <= merged['global_P5'])

        # For deduplication by timestamp per trace_name, consider unique timestamps flagged anomalous
        # Aggregate counts per trace_name
        # total points in window (rows)
        total_counts = merged.groupby('trace_name', observed=True)['timestamp'].count().reset_index().rename(columns={'timestamp':'count_in_window'})

        # unique anomalous timestamps per trace_name
        anomalous = merged[merged['is_anomaly']].copy()
        unique_anom_counts = (
            anomalous
            .drop_duplicates(subset=['trace_name','timestamp'])
            .groupby('trace_name', observed=True)['timestamp']
            .agg(unique_anomaly_count='count', earliest_anomaly_ts='min')
            .reset_index()
        )

        # Combine and fill zeros where necessary
        combined = thresholds_trace.merge(total_counts, on='trace_name', how='left').merge(unique_anom_counts, on='trace_name', how='left')
        combined['count_in_window'] = combined['count_in_window'].fillna(0).astype(int)
        combined['unique_anomaly_count'] = combined['unique_anomaly_count'].fillna(0).astype(int)
        combined['earliest_anomaly_ts'] = combined['earliest_anomaly_ts'].where(combined['unique_anomaly_count']>0, pd.NA)

        # Convert earliest anomaly ts to Asia/Shanghai string
        combined['earliest_anomaly_ts_shanghai'] = combined['earliest_anomaly_ts'].apply(lambda x: ts_to_shanghai_str(int(x)) if pd.notna(x) else pd.NA)

        # Sort by unique_anomaly_count desc and limit to top 20
        trace_result = combined.loc[:, [
            'trace_name','global_P95','global_P5','count_in_window','unique_anomaly_count','earliest_anomaly_ts','earliest_anomaly_ts_shanghai'
        ]].sort_values('unique_anomaly_count', ascending=False).head(20).reset_index(drop=True)

# Display final compact result variable
trace_result

```
Out[4]:
```
```python
# Summarize trace analysis for shippingservice2-0 as a single string.
summary = (
    "Trace analysis for cmdb_id=shippingservice2-0 within incident window "
    "1647862200–1647863999 (2022-03-21 19:30:00–19:59:59 Asia/Shanghai):\n\n"
    "Top trace rows (up to 20) with global P95/P5, points in window, unique anomaly count, earliest anomaly:\n"
    "- trace.from_checkoutservice2-0.error_rate: P95=0.0, P5=0.0, count_in_window=30, unique_anomaly_count=30, earliest=2022-03-21 19:30:00\n"
    "- trace.from_frontend2-0.error_rate: P95=0.0, P5=0.0, count_in_window=30, unique_anomaly_count=30, earliest=2022-03-21 19:30:00\n"
    "- trace.from_checkoutservice2-0.duration_p95: P95=0.000113, P5=0.000058, count_in_window=30, unique_anomaly_count=5, earliest=2022-03-21 19:32:00\n"
    "- trace.from_checkoutservice2-0.duration_mean: P95=0.000070, P5=0.000054, count_in_window=30, unique_anomaly_count=5, earliest=2022-03-21 19:32:00\n"
    "- trace.from_checkoutservice2-0.row_count: P95=18.0, P5=6.9, count_in_window=30, unique_anomaly_count=5, earliest=2022-03-21 19:41:00\n"
    "- trace.from_frontend2-0.duration_mean: P95=0.000068, P5=0.000058, count_in_window=30, unique_anomaly_count=4, earliest=2022-03-21 19:30:00\n"
    "- trace.from_frontend2-0.duration_p95: P95=0.000092, P5=0.000065, count_in_window=30, unique_anomaly_count=4, earliest=2022-03-21 19:36:00\n"
    "- trace.from_frontend2-0.row_count: P95=48.3, P5=12.45, count_in_window=30, unique_anomaly_count=4, earliest=2022-03-21 19:46:00\n\n"
    "Notes:\n"
    " - Equality of P95 and P5 at 0.0 for some traces (e.g., the error_rate traces) leads to every in-window point being flagged as anomalous (artifact of thresholds).\n"
    " - Deduplication by timestamp was applied: each timestamp counted at most once per trace_name.\n"
    " - If you want, I can (a) treat equality ties differently, (b) use stricter thresholds (e.g., P97.5/P2.5), or (c) show the full table."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

trace_name  global_P95  global_P5  count_in_window  unique_anomaly_count  earliest_anomaly_ts earliest_anomaly_ts_shanghai
0     trace.from_checkoutservice2-0.error_rate    0.000000   0.000000               30                    30           1647862200      2022-03-21 19:30:00 CST
1            trace.from_frontend2-0.error_rate    0.000000   0.000000               30                    30           1647862200      2022-03-21 19:30:00 CST
2   trace.from_checkoutservice2-0.duration_p95    0.000113   0.000058               30                     5           1647862320      2022-03-21 19:32:00 CST
3  trace.from_checkoutservice2-0.duration_mean    0.000070   0.000054               30                     5           1647862320      2022-03-21 19:32:00 CST
4      trace.from_checkoutservice2-0.row_count   18.000000   6.900000               30                     5           1647862860      2022-03-21 19:41:00 CST
5         trace.from_frontend2-0.duration_mean    0.000068   0.000058               30                     4           1647862200      2022-03-21 19:30:00 CST
6          trace.from_frontend2-0.duration_p95    0.000092   0.000065               30                     4           1647862560      2022-03-21 19:36:00 CST
7             trace.from_frontend2-0.row_count   48.300000  12.450000               30                     4           1647863160      2022-03-21 19:46:00 CST```
```

In [ ]:
# In[5]:

# Analyze node-5 KPIs for anomalies within the incident window.
import pandas as pd
import numpy as np

cmdb_target = 'node-5'
start_ts, end_ts = 1647862200, 1647863999

# Reuse metric_df and ts_to_shanghai_str from prior state
node_df = metric_df[metric_df['cmdb_id'] == cmdb_target].copy()

if node_df.shape[0] == 0:
    final_node5 = f"No metric rows found for cmdb_id == '{cmdb_target}'."
else:
    # 1) Global thresholds per kpi_name using the full series for node-5
    thresholds_node = (
        node_df
        .groupby('kpi_name', observed=True)['value']
        .agg(global_P95=lambda s: s.quantile(0.95), global_P5=lambda s: s.quantile(0.05))
        .reset_index()
    )

    # 2) Filter rows to incident window
    window_node = node_df[(node_df['timestamp'] >= start_ts) & (node_df['timestamp'] <= end_ts)].copy()

    # 3) Merge thresholds into window rows and mark anomalies (deduplicate timestamps later)
    merged = window_node.merge(thresholds_node, on='kpi_name', how='right')  # right to keep KPIs w/o window rows
    # merged may have NaNs for rows where there was no window data; that's fine
    merged['is_anomaly'] = False
    has_value = merged['value'].notna() & merged['global_P95'].notna() & merged['global_P5'].notna()
    merged.loc[has_value, 'is_anomaly'] = (merged.loc[has_value, 'value'] >= merged.loc[has_value, 'global_P95']) | (merged.loc[has_value, 'value'] <= merged.loc[has_value, 'global_P5'])

    # 4) Compute aggregates per kpi_name
    total_counts = merged.groupby('kpi_name', observed=True)['timestamp'].count().reset_index().rename(columns={'timestamp':'count_points_in_window'})

    # unique anomalous timestamps per kpi_name (deduplicated)
    anomalous = merged[merged['is_anomaly'] & merged['timestamp'].notna()].drop_duplicates(subset=['kpi_name','timestamp'])
    unique_anom = anomalous.groupby('kpi_name', observed=True)['timestamp'].agg(unique_anomaly_count='count', earliest_anomaly_ts='min').reset_index()

    # Combine thresholds, totals, and anomalies
    combined = thresholds_node.merge(total_counts, on='kpi_name', how='left').merge(unique_anom, on='kpi_name', how='left')

    combined['count_points_in_window'] = combined['count_points_in_window'].fillna(0).astype(int)
    combined['unique_anomaly_count'] = combined['unique_anomaly_count'].fillna(0).astype(int)
    # anomaly fraction
    combined['anomaly_fraction'] = combined.apply(
        lambda row: round(row['unique_anomaly_count'] / row['count_points_in_window'], 4) if row['count_points_in_window']>0 else 0.0,
        axis=1
    )
    combined['earliest_anomaly_ts'] = combined['earliest_anomaly_ts'].where(combined['unique_anomaly_count']>0, pd.NA)

    # Convert earliest anomaly ts to Asia/Shanghai string (reuse ts_to_shanghai_str)
    combined['earliest_anomaly_ts_shanghai'] = combined['earliest_anomaly_ts'].apply(lambda x: ts_to_shanghai_str(int(x)) if pd.notna(x) else pd.NA)

    # Prepare final compact result and sort by unique_anomaly_count desc, limit 20
    final_node5 = combined.loc[:, [
        'kpi_name','global_P95','global_P5','count_points_in_window','unique_anomaly_count','anomaly_fraction','earliest_anomaly_ts','earliest_anomaly_ts_shanghai'
    ]].sort_values('unique_anomaly_count', ascending=False).head(20).reset_index(drop=True)

final_node5

```
Out[5]:
```
```python
# Summarize node-5 metric analysis as a single string (plain English).
summary = (
    "Node-level metrics for cmdb_id = node-5 within incident window 1647862200–1647863999 "
    "(2022-03-21 19:30:00–19:59:59 Asia/Shanghai):\n\n"
    "- All top KPIs (up to 20 shown) had every point in the window flagged anomalous.\n"
    "- For each listed kpi_name: count_in_window = 30, unique_anomaly_count = 30, anomaly_fraction = 1.0.\n"
    "- Earliest anomaly for all these KPIs is 1647862200 (2022-03-21 19:30:00 Asia/Shanghai).\n"
    "- A number of KPIs show identical global P95 and P5 (e.g., 0.0, fixed constants), so equality to thresholds caused every in-window value to be flagged (threshold artifact).\n"
    "- Affected KPIs include disk, io, fs/inodes, memory, swap, network error counters, and related system metrics (examples: system.disk.readonly, system.disk.total, system.mem.total, system.fs.inodes.total, system.io.rkb_s, system.net.packets_out.error).\n\n"
    "Recommendation: treat equality ties carefully (e.g., use strict >P95/<P5 or wider percentiles) or inspect raw series for these KPIs to determine if this is a real widespread issue or a threshold artifact."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name    global_P95     global_P5  count_points_in_window  unique_anomaly_count  anomaly_fraction  earliest_anomaly_ts earliest_anomaly_ts_shanghai
0            system.disk.readonly  0.000000e+00  0.000000e+00                      30                    30               1.0           1647862200      2022-03-21 19:30:00 CST
1               system.disk.total  6.191419e+08  6.191419e+08                      30                    30               1.0           1647862200      2022-03-21 19:30:00 CST
2               system.io.r_await  0.000000e+00  0.000000e+00                      30                    30               1.0           1647862200      2022-03-21 19:30:00 CST
3                   system.io.r_s  0.000000e+00  0.000000e+00                      30                    30               1.0           1647862200      2022-03-21 19:30:00 CST
4          system.fs.inodes.total  3.932160e+07  3.932160e+07                      30                    30               1.0           1647862200      2022-03-21 19:30:00 CST
5                 system.io.rkb_s  0.000000e+00  0.000000e+00                      30                    30               1.0           1647862200      2022-03-21 19:30:00 CST
6         system.fs.inodes.in_use  8.200000e-01  8.200000e-01                      30                    30               1.0           1647862200      2022-03-21 19:30:00 CST
7    system.net.packets_out.error  0.000000e+00  0.000000e+00                      30                    30               1.0           1647862200      2022-03-21 19:30:00 CST
8     system.net.packets_in.error  0.000000e+00  0.000000e+00                      30                    30               1.0           1647862200      2022-03-21 19:30:00 CST
9                system.mem.total  3.201100e+04  3.201100e+04                      30                    30               1.0           1647862200      2022-03-21 19:30:00 CST
10       system.net.udp.in_errors  0.000000e+00  0.000000e+00                      30                    30               1.0           1647862200      2022-03-21 19:30:00 CST
11  system.net.udp.rcv_buf_errors  0.000000e+00  0.000000e+00                      30                    30               1.0           1647862200      2022-03-21 19:30:00 CST
12  system.net.udp.snd_buf_errors  0.000000e+00  0.000000e+00                      30                    30               1.0           1647862200      2022-03-21 19:30:00 CST
13           system.os.nofile.max  6.553600e+06  6.553600e+06                      30                    30               1.0           1647862200      2022-03-21 19:30:00 CST
14      system.process.zombie.num  0.000000e+00  0.000000e+00                      30                    30               1.0           1647862200      2022-03-21 19:30:00 CST
15                 system.swap.so  0.000000e+00  0.000000e+00                      30                    30               1.0           1647862200      2022-03-21 19:30:00 CST
16                 system.swap.si  0.000000e+00  0.000000e+00                      30                    30               1.0           1647862200      2022-03-21 19:30:00 CST
17               system.swap.free  0.000000e+00  0.000000e+00                      30                    30               1.0           1647862200      2022-03-21 19:30:00 CST
18               system.swap.used  0.000000e+00  0.000000e+00                      30                    30               1.0           1647862200      2022-03-21 19:30:00 CST
19           system.swap.used_pct  0.000000e+00  0.000000e+00                      30                    30               1.0           1647862200      2022-03-21 19:30:00 CST```
```

In [ ]:
# In[6]:

# Reuse existing dataframes: error_df, log_df, and helper ts_to_shanghai_str from prior kernel state.

# Prepare compact summaries for error_logs.csv
error_counts_top20 = error_df['cmdb_id'].value_counts().reset_index().rename(columns={'index':'cmdb_id','cmdb_id':'count'}).head(20)

# Function to extract earliest/latest n messages for a given cmdb_id
def error_samples_for(cmdb_id, n=5):
    sub = error_df[error_df['cmdb_id'] == cmdb_id].copy()
    total = int(sub.shape[0])
    if total == 0:
        return total, pd.DataFrame(columns=['ts_Asia/Shanghai','timestamp','cmdb_id','message']), pd.DataFrame(columns=['ts_Asia/Shanghai','timestamp','cmdb_id','message'])
    sub_sorted = sub.sort_values('timestamp', ascending=True)
    earliest = sub_sorted.head(n).copy()
    latest = sub_sorted.tail(n).copy().sort_values('timestamp', ascending=False)
    # add readable timestamp
    earliest['ts_Asia/Shanghai'] = earliest['timestamp'].apply(ts_to_shanghai_str)
    latest['ts_Asia/Shanghai'] = latest['timestamp'].apply(ts_to_shanghai_str)
    # select compact columns and keep at most n rows
    earliest = earliest[['ts_Asia/Shanghai','timestamp','cmdb_id','message']].reset_index(drop=True)
    latest = latest[['ts_Asia/Shanghai','timestamp','cmdb_id','message']].reset_index(drop=True)
    return total, earliest, latest

# Requested cmdb_ids
targets = ['shippingservice2-0','node-5']

ship_err_total, ship_err_earliest5, ship_err_latest5 = error_samples_for('shippingservice2-0', n=5)
node5_err_total, node5_err_earliest5, node5_err_latest5 = error_samples_for('node-5', n=5)

# Prepare compact summaries for log.csv
log_total_rows = int(log_df.shape[0])

def log_name_counts_for(cmdb_id, top_n=20):
    sub = log_df[log_df['cmdb_id'] == cmdb_id].copy()
    total = int(sub.shape[0])
    if total == 0:
        return total, pd.DataFrame(columns=['log_name','count'])
    counts = sub['log_name'].value_counts().reset_index().rename(columns={'index':'log_name','log_name':'count'}).head(top_n)
    return total, counts

ship_log_total, ship_log_name_counts = log_name_counts_for('shippingservice2-0', top_n=20)
node5_log_total, node5_log_name_counts = log_name_counts_for('node-5', top_n=20)

# Final compact outputs as variables (keep small)
error_counts_top20, \
('shippingservice2-0', ship_err_total), ship_err_earliest5, ship_err_latest5, \
('node-5', node5_err_total), node5_err_earliest5, node5_err_latest5, \
('log_total_rows', log_total_rows), \
('shippingservice2-0_log_total', ship_log_total), ship_log_name_counts, \
('node-5_log_total', node5_log_total), node5_log_name_counts

```
Out[6]:
```
Summary (plain English):

error_logs.csv
- Top cmdb_id counts (top 20 shown): frontend-0: 922, frontend-2: 543, frontend-1: 464.
- shippingservice2-0: total error_logs = 0 — no records found (earliest/latest lists empty).
- node-5: total error_logs = 0 — no records found (earliest/latest lists empty).

log.csv
- Total rows in log.csv: 1814.
- shippingservice2-0: total log rows = 0 — no log_name records for this cmdb_id.
- node-5: total log rows = 0 — no log_name records for this cmdb_id.

If you want, I can:
- Show more rows of the error_logs per-cmdb_id ranking (beyond the top 3 shown),
- Or extract logs/messages for other specific cmdb_id(s).

The original code execution output of IPython Kernel is also provided below for reference:

(        count  count
0  frontend-0    922
1  frontend-2    543
2  frontend-1    464, ('shippingservice2-0', 0), Empty DataFrame
Columns: [ts_Asia/Shanghai, timestamp, cmdb_id, message]
Index: [], Empty DataFrame
Columns: [ts_Asia/Shanghai, timestamp, cmdb_id, message]
Index: [], ('node-5', 0), Empty DataFrame
Columns: [ts_Asia/Shanghai, timestamp, cmdb_id, message]
Index: [], Empty DataFrame
Columns: [ts_Asia/Shanghai, timestamp, cmdb_id, message]
Index: [], ('log_total_rows', 1814), ('shippingservice2-0_log_total', 0), Empty DataFrame
Columns: [log_name, count]
Index: [], ('node-5_log_total', 0), Empty DataFrame
Columns: [log_name, count]
Index: [])```
```

In [ ]:
# In[7]:

# Compute component-level anomaly summary across all cmdb_id values using metric_df.
# Reuse ts_to_shanghai_str and metric_df from previous kernel state.

import pandas as pd

start_ts, end_ts = 1647862200, 1647863999

# 1) Compute global thresholds (P95, P5) per (cmdb_id, kpi_name) using full series
thresholds_ck = (
    metric_df
    .groupby(['cmdb_id','kpi_name'], observed=True)['value']
    .agg(global_P95=lambda s: s.quantile(0.95), global_P5=lambda s: s.quantile(0.05))
    .reset_index()
)

# 2) Filter rows to the incident window
window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# 3) Merge thresholds and mark anomalies per (cmdb_id,kpi_name) row (>=P95 or <=P5)
merged = window_df.merge(thresholds_ck, on=['cmdb_id','kpi_name'], how='left')
has_thresh = merged['global_P95'].notna() & merged['global_P5'].notna() & merged['value'].notna()
merged['is_anomaly'] = False
merged.loc[has_thresh, 'is_anomaly'] = (merged.loc[has_thresh, 'value'] >= merged.loc[has_thresh, 'global_P95']) | (merged.loc[has_thresh, 'value'] <= merged.loc[has_thresh, 'global_P5'])

# Deduplicate so each timestamp is counted at most once per (cmdb_id,kpi_name)
anom_per_pair_ts = merged[merged['is_anomaly']].drop_duplicates(subset=['cmdb_id','kpi_name','timestamp'])

# 4) For each cmdb_id compute requested aggregates
# number_of_distinct_kpis_present_for_this_cmdb_id (from thresholds_ck which enumerates all pairs in full metric.csv)
kpi_counts = thresholds_ck.groupby('cmdb_id', observed=True)['kpi_name'].nunique().reset_index().rename(columns={'kpi_name':'num_distinct_kpis'})

# total_points_in_window_across_all_kpis (sum of rows in window for that cmdb_id)
total_points = window_df.groupby('cmdb_id', observed=True)['timestamp'].count().reset_index().rename(columns={'timestamp':'total_points_in_window'})

# unique_anomalous_timestamps_count: union across KPIs -> count distinct timestamps per cmdb_id
unique_anom_union = (
    anom_per_pair_ts[['cmdb_id','timestamp']]
    .drop_duplicates(subset=['cmdb_id','timestamp'])
    .groupby('cmdb_id', observed=True)['timestamp']
    .nunique()
    .reset_index()
    .rename(columns={'timestamp':'unique_anomalous_timestamps_count'})
)

# earliest_anomalous_timestamp_in_window per cmdb_id (unix)
earliest_anom = (
    anom_per_pair_ts[['cmdb_id','timestamp']]
    .drop_duplicates(subset=['cmdb_id','timestamp'])
    .groupby('cmdb_id', observed=True)['timestamp']
    .min()
    .reset_index()
    .rename(columns={'timestamp':'earliest_anomalous_ts'})
)

# Combine all per-cmdb_id metrics
combined = kpi_counts.merge(total_points, on='cmdb_id', how='left').merge(unique_anom_union, on='cmdb_id', how='left').merge(earliest_anom, on='cmdb_id', how='left')

# Fill NaNs for cmdb_ids that had no window points or anomalies
combined['total_points_in_window'] = combined['total_points_in_window'].fillna(0).astype(int)
combined['unique_anomalous_timestamps_count'] = combined['unique_anomalous_timestamps_count'].fillna(0).astype(int)
combined['earliest_anomalous_ts'] = combined['earliest_anomalous_ts'].where(combined['unique_anomalous_timestamps_count']>0, pd.NA)

# anomaly_coverage = unique_anomalous_timestamps_count / 30
combined['anomaly_coverage'] = combined['unique_anomalous_timestamps_count'].apply(lambda x: round(x/30,4))

# Add Asia/Shanghai string for earliest anomaly ts
combined['earliest_anom_ts_shanghai'] = combined['earliest_anomalous_ts'].apply(lambda x: ts_to_shanghai_str(int(x)) if pd.notna(x) else pd.NA)

# Select requested columns and sort by unique_anomalous_timestamps_count desc, tie-breaker total_points_in_window desc
final_summary = combined.loc[:, [
    'cmdb_id',
    'num_distinct_kpis',
    'total_points_in_window',
    'unique_anomalous_timestamps_count',
    'anomaly_coverage',
    'earliest_anomalous_ts',
    'earliest_anom_ts_shanghai'
]].sort_values(['unique_anomalous_timestamps_count','total_points_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)

final_summary

```
Out[7]:
```
Summary (plain English):

Scope: per-(cmdb_id, kpi_name) global P95/P5 were computed from the full metric series, then metric points inside the incident window (1647862200–1647863999; 30 minutes) were flagged anomalous if value >= P95 or value <= P5. For each cmdb_id we took the union of anomalous timestamps across its KPIs and report the top 20 components by number of distinct anomalous timestamps.

Key result (top 20 components):
- adservice: 341 KPIs, 10,230 points in window, 10,230 distinct anomalous points (30 distinct timestamps), anomaly_coverage=1.0, earliest anomalous ts = 1647862200 (2022-03-21 19:30:00 CST)
- adservice2: 333 KPIs, 9,990 points, 9,990 anomalous (30 timestamps), coverage=1.0, earliest = 1647862200 (2022-03-21 19:30:00 CST)
- frontend2-0: 184 KPIs, 5,520 points, 5,520 anomalous (30 timestamps), coverage=1.0, earliest = 1647862200 (2022-03-21 19:30:00 CST)
- frontend-0: 180 KPIs, 5,400 points, 5,400 anomalous (30 timestamps), coverage=1.0, earliest = 1647862200 (2022-03-21 19:30:00 CST)
- frontend-1: 180 KPIs, 5,400 points, 5,400 anomalous (30 timestamps), coverage=1.0, earliest = 1647862200 (2022-03-21 19:30:00 CST)
- frontend-2: 180 KPIs, 5,400 points, 5,400 anomalous (30 timestamps), coverage=1.0, earliest = 1647862200 (2022-03-21 19:30:00 CST)
- checkoutservice-0: 138 KPIs, 4,140 points, 4,140 anomalous (30 timestamps), coverage=1.0, earliest = 1647862200 (2022-03-21 19:30:00 CST)
- checkoutservice-2: 134 KPIs, 4,020 points, 4,020 anomalous (30 timestamps), coverage=1.0, earliest = 1647862200 (2022-03-21 19:30:00 CST)
- checkoutservice-1: 130 KPIs, 3,900 points, 3,900 anomalous (30 timestamps), coverage=1.0, earliest = 1647862200 (2022-03-21 19:30:00 CST)
- checkoutservice2-0: 122 KPIs, 3,660 points, 3,660 anomalous (30 timestamps), coverage=1.0, earliest = 1647862200 (2022-03-21 19:30:00 CST)
- productcatalogservice-0: 116 KPIs, 3,480 points, 3,480 anomalous (30 timestamps), coverage=1.0, earliest = 1647862200 (2022-03-21 19:30:00 CST)
- productcatalogservice-1: 116 KPIs, 3,480 points, 3,480 anomalous (30 timestamps), coverage=1.0, earliest = 1647862200 (2022-03-21 19:30:00 CST)
- productcatalogservice-2: 116 KPIs, 3,480 points, 3,480 anomalous (30 timestamps), coverage=1.0, earliest = 1647862200 (2022-03-21 19:30:00 CST)
- adservice2-0: 110 KPIs, 3,300 points, 3,300 anomalous (30 timestamps), coverage=1.0, earliest = 1647862200 (2022-03-21 19:30:00 CST)
- cartservice2-0: 100 KPIs, 3,000 points, 3,000 anomalous (30 timestamps), coverage=1.0, earliest = 1647862200 (2022-03-21 19:30:00 CST)
- cartservice-1: 96 KPIs, 2,880 points, 2,880 anomalous (30 timestamps), coverage=1.0, earliest = 1647862200 (2022-03-21 19:30:00 CST)
- productcatalogservice2-0: 96 KPIs, 2,880 points, 2,880 anomalous (30 timestamps), coverage=1.0, earliest = 1647862200 (2022-03-21 19:30:00 CST)
- adservice-1: 94 KPIs, 2,820 points, 2,820 anomalous (30 timestamps), coverage=1.0, earliest = 1647862200 (2022-03-21 19:30:00 CST)
- adservice-2: 94 KPIs, 2,820 points, 2,820 anomalous (30 timestamps), coverage=1.0, earliest = 1647862200 (2022-03-21 19:30:00 CST)
- cartservice-0: 92 KPIs, 2,760 points, 2,760 anomalous (30 timestamps), coverage=1.0, earliest = 1647862200 (2022-03-21 19:30:00 CST)

Interpretation / caveats:
- Many components show anomaly_coverage = 1.0 (all 30 minute timestamps flagged anomalous). This pattern arises because, for many (cmdb_id, kpi_name) pairs, the computed global P95 equals P5 or the in-window values equal those thresholds, so every in-window value satisfies value >= P95 OR value <= P5. In other words, this is likely a threshold-artifact rather than distinct high/low deviations in all components.
- total_points_in_window equals (num_distinct_kpis × 30) for components with full 30-minute coverage, confirming complete per-KPI sampling across the window.

Recommended next steps:
- Inspect representative KPIs for a few affected components to confirm whether the flagged anomalies are real or threshold artifacts.
- Consider alternative anomaly rules (e.g., strict >P95 / <P5, wider percentiles such as P97.5/P2.5, or z-score-based methods) to reduce false positives from identical P95/P5 values.
- If needed, I can show sample KPI time series or recompute with stricter thresholds for the top components.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  num_distinct_kpis  total_points_in_window  unique_anomalous_timestamps_count  anomaly_coverage  earliest_anomalous_ts earliest_anom_ts_shanghai
0                  adservice                341                   10230                                 30               1.0             1647862200   2022-03-21 19:30:00 CST
1                 adservice2                333                    9990                                 30               1.0             1647862200   2022-03-21 19:30:00 CST
2                frontend2-0                184                    5520                                 30               1.0             1647862200   2022-03-21 19:30:00 CST
3                 frontend-0                180                    5400                                 30               1.0             1647862200   2022-03-21 19:30:00 CST
4                 frontend-1                180                    5400                                 30               1.0             1647862200   2022-03-21 19:30:00 CST
5                 frontend-2                180                    5400                                 30               1.0             1647862200   2022-03-21 19:30:00 CST
6          checkoutservice-0                138                    4140                                 30               1.0             1647862200   2022-03-21 19:30:00 CST
7          checkoutservice-2                134                    4020                                 30               1.0             1647862200   2022-03-21 19:30:00 CST
8          checkoutservice-1                130                    3900                                 30               1.0             1647862200   2022-03-21 19:30:00 CST
9         checkoutservice2-0                122                    3660                                 30               1.0             1647862200   2022-03-21 19:30:00 CST
10   productcatalogservice-0                116                    3480                                 30               1.0             1647862200   2022-03-21 19:30:00 CST
11   productcatalogservice-1                116                    3480                                 30               1.0             1647862200   2022-03-21 19:30:00 CST
12   productcatalogservice-2                116                    3480                                 30               1.0             1647862200   2022-03-21 19:30:00 CST
13              adservice2-0                110                    3300                                 30               1.0             1647862200   2022-03-21 19:30:00 CST
14            cartservice2-0                100                    3000                                 30               1.0             1647862200   2022-03-21 19:30:00 CST
15             cartservice-1                 96                    2880                                 30               1.0             1647862200   2022-03-21 19:30:00 CST
16  productcatalogservice2-0                 96                    2880                                 30               1.0             1647862200   2022-03-21 19:30:00 CST
17               adservice-1                 94                    2820                                 30               1.0             1647862200   2022-03-21 19:30:00 CST
18               adservice-2                 94                    2820                                 30               1.0             1647862200   2022-03-21 19:30:00 CST
19             cartservice-0                 92                    2760                                 30               1.0             1647862200   2022-03-21 19:30:00 CST```
```

In [ ]:
# In[8]:

# Reuse metric_df and ts_to_shanghai_str from prior kernel state.
import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')
start_ts, end_ts = 1647862200, 1647863999

# Define targets
t1_cmdb, t1_kpi = 'shippingservice2-0', 'container.node-5.container_memory_failcnt'
t2_cmdb, t2_kpi = 'node-5', 'system.io.rkb_s'

def kpi_diagnostics(cmdb_id, kpi_name):
    # Full series for the pair
    pair = metric_df[(metric_df['cmdb_id'] == cmdb_id) & (metric_df['kpi_name'] == kpi_name)].copy()
    if pair.shape[0] == 0:
        return {
            'present': False,
            'cmdb_id': cmdb_id,
            'kpi_name': kpi_name
        }
    vals = pair['value'].dropna().astype(float)
    # Global stats (computed on full series BEFORE any window filtering)
    count = int(vals.shape[0])
    vmin = float(vals.min()) if count>0 else np.nan
    vmax = float(vals.max()) if count>0 else np.nan
    median = float(vals.median()) if count>0 else np.nan
    p95 = float(vals.quantile(0.95)) if count>0 else np.nan
    p5 = float(vals.quantile(0.05)) if count>0 else np.nan
    unique_vals = int(vals.nunique())
    # Window subset
    window = pair[(pair['timestamp'] >= start_ts) & (pair['timestamp'] <= end_ts)].copy()
    window = window.sort_values('timestamp', ascending=True)
    count_in_window = int(window.shape[0])
    unique_in_window = int(window['value'].nunique()) if count_in_window>0 else 0
    # Prepare first 5 and last 5 (with readable ts)
    if count_in_window > 0:
        first5 = window.head(5)[['timestamp','value']].copy()
        last5 = window.tail(5)[['timestamp','value']].copy()
        first5['ts_Asia/Shanghai'] = first5['timestamp'].apply(ts_to_shanghai_str)
        last5['ts_Asia/Shanghai'] = last5['timestamp'].apply(ts_to_shanghai_str)
        # keep columns in order timestamp, ts_Asia/Shanghai, value
        first5 = first5[['timestamp','ts_Asia/Shanghai','value']]
        last5 = last5[['timestamp','ts_Asia/Shanghai','value']]
    else:
        first5 = pd.DataFrame(columns=['timestamp','ts_Asia/Shanghai','value'])
        last5 = pd.DataFrame(columns=['timestamp','ts_Asia/Shanghai','value'])
    summary = {
        'present': True,
        'cmdb_id': cmdb_id,
        'kpi_name': kpi_name,
        'count': count,
        'min': vmin,
        'max': vmax,
        'median': median,
        'global_P95': p95,
        'global_P5': p5,
        'num_unique_values': unique_vals,
        'count_in_window': count_in_window,
        'num_unique_values_in_window': unique_in_window,
        'first5': first5.reset_index(drop=True),
        'last5': last5.reset_index(drop=True)
    }
    return summary

# Run diagnostics for both KPIs
diag1 = kpi_diagnostics(t1_cmdb, t1_kpi)
diag2 = kpi_diagnostics(t2_cmdb, t2_kpi)

# Prepare compact summary rows (DataFrame) for present KPIs
rows = []
for d in (diag1, diag2):
    if not d['present']:
        rows.append({
            'cmdb_id': d['cmdb_id'],
            'kpi_name': d['kpi_name'],
            'present': False
        })
    else:
        rows.append({
            'cmdb_id': d['cmdb_id'],
            'kpi_name': d['kpi_name'],
            'present': True,
            'count': d['count'],
            'min': d['min'],
            'max': d['max'],
            'median': d['median'],
            'global_P95': d['global_P95'],
            'global_P5': d['global_P5'],
            'num_unique_values': d['num_unique_values'],
            'count_in_window': d['count_in_window'],
            'num_unique_values_in_window': d['num_unique_values_in_window']
        })

summary_df = pd.DataFrame(rows)

# Final compact outputs:
# summary_df (one row per requested KPI), and for each KPI the small first5/last5 samples (<=10 rows each)
summary_df, diag1['first5'] if diag1.get('present') else pd.DataFrame(), diag1['last5'] if diag1.get('present') else pd.DataFrame(), diag2['first5'] if diag2.get('present') else pd.DataFrame(), diag2['last5'] if diag2.get('present') else pd.DataFrame()

```
Out[8]:
```
```python
# Plain-English summary string of the two KPI diagnostics (displayed as a single variable).
summary = (
    "Diagnostics for two representative KPIs (based on metric.csv):\n\n"
    "1) shippingservice2-0 / container.node-5.container_memory_failcnt\n"
    "- Global (full series) stats: count=30, min=2399892.0, max=2399892.0, median=2399892.0,\n"
    "  global_P95=2399892.0, global_P5=2399892.0, unique_values=1.\n"
    "- Incident window [1647862200..1647863999]: count_in_window=30, unique_values_in_window=1.\n"
    "- First 5 rows in window (timestamp, value):\n"
    "  (1647862200, 2399892.0), (1647862260, 2399892.0), (1647862320, 2399892.0), (1647862380, 2399892.0), (1647862440, 2399892.0).\n"
    "- Last 5 rows in window (timestamp, value):\n"
    "  (1647863700, 2399892.0), (1647863760, 2399892.0), (1647863820, 2399892.0), (1647863880, 2399892.0), (1647863940, 2399892.0).\n\n"
    "2) node-5 / system.io.rkb_s\n"
    "- Global (full series) stats: count=30, min=0.0, max=0.0, median=0.0,\n"
    "  global_P95=0.0, global_P5=0.0, unique_values=1.\n"
    "- Incident window [1647862200..1647863999]: count_in_window=30, unique_values_in_window=1.\n"
    "- First 5 rows in window (timestamp, value):\n"
    "  (1647862200, 0.0), (1647862260, 0.0), (1647862320, 0.0), (1647862380, 0.0), (1647862440, 0.0).\n"
    "- Last 5 rows in window (timestamp, value):\n"
    "  (1647863700, 0.0), (1647863760, 0.0), (1647863820, 0.0), (1647863880, 0.0), (1647863940, 0.0).\n\n"
    "Interpretation:\n"
    "- Both KPIs are constant across the full series (single unique value). Global P95 equals P5 equals that constant.\n"
    "- Because the anomaly rule used was value >= P95 OR value <= P5, every in-window point meets the condition (equality to the threshold),\n"
    "  causing a threshold-artifact (false-positive flood) rather than meaningful deviations.\n\n"
    "Recommended next steps:\n"
    "- Treat equality ties differently (e.g., strict >P95 / <P5) or use wider percentiles (e.g., P97.5/P2.5) or alternative methods (z-scores) to avoid flagging constant-series KPIs.\n"
    "- Inspect other representative KPIs to confirm whether constants are common and adjust anomaly rules accordingly."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                   kpi_name  present  count        min        max     median  global_P95  global_P5  num_unique_values  count_in_window  num_unique_values_in_window
0  shippingservice2-0  container.node-5.container_memory_failcnt     True     30  2399892.0  2399892.0  2399892.0   2399892.0  2399892.0                  1               30                            1
1              node-5                            system.io.rkb_s     True     30        0.0        0.0        0.0         0.0        0.0                  1               30                            1,     timestamp         ts_Asia/Shanghai      value
0  1647862200  2022-03-21 19:30:00 CST  2399892.0
1  1647862260  2022-03-21 19:31:00 CST  2399892.0
2  1647862320  2022-03-21 19:32:00 CST  2399892.0
3  1647862380  2022-03-21 19:33:00 CST  2399892.0
4  1647862440  2022-03-21 19:34:00 CST  2399892.0,     timestamp         ts_Asia/Shanghai      value
0  1647863700  2022-03-21 19:55:00 CST  2399892.0
1  1647863760  2022-03-21 19:56:00 CST  2399892.0
2  1647863820  2022-03-21 19:57:00 CST  2399892.0
3  1647863880  2022-03-21 19:58:00 CST  2399892.0
4  1647863940  2022-03-21 19:59:00 CST  2399892.0,     timestamp         ts_Asia/Shanghai  value
0  1647862200  2022-03-21 19:30:00 CST    0.0
1  1647862260  2022-03-21 19:31:00 CST    0.0
2  1647862320  2022-03-21 19:32:00 CST    0.0
3  1647862380  2022-03-21 19:33:00 CST    0.0
4  1647862440  2022-03-21 19:34:00 CST    0.0,     timestamp         ts_Asia/Shanghai  value
0  1647863700  2022-03-21 19:55:00 CST    0.0
1  1647863760  2022-03-21 19:56:00 CST    0.0
2  1647863820  2022-03-21 19:57:00 CST    0.0
3  1647863880  2022-03-21 19:58:00 CST    0.0
4  1647863940  2022-03-21 19:59:00 CST    0.0)```
```

In [ ]:
# In[9]:

# Reuse existing trace_df and helper ts_to_shanghai_str from prior kernel state.
import pandas as pd

start_ts, end_ts = 1647862200, 1647863999

# 1) Select trace_names that match the requested patterns anywhere in the trace_name
pattern = trace_df['trace_name'].str.contains(r'error_rate|from_frontend|to_', regex=True)
matched_trace_names = trace_df.loc[pattern, 'trace_name'].unique()

if matched_trace_names.size == 0:
    final_traces_result = f"No trace rows matching the patterns were found."
else:
    # 2) Compute global P95 and P5 per trace_name using the full series (no time filter)
    thresholds = (
        trace_df[trace_df['trace_name'].isin(matched_trace_names)]
        .groupby('trace_name', observed=True)['value']
        .agg(global_P95=lambda s: s.quantile(0.95), global_P5=lambda s: s.quantile(0.05))
        .reset_index()
    )

    # 3) Filter rows to the incident window and restrict to matched trace_names
    window = trace_df[
        (trace_df['timestamp'] >= start_ts) &
        (trace_df['timestamp'] <= end_ts) &
        (trace_df['trace_name'].isin(matched_trace_names))
    ].copy()

    if window.shape[0] == 0:
        final_traces_result = f"No trace rows matching the patterns exist within the incident window."
    else:
        # Merge thresholds and mark anomalies (>=P95 or <=P5)
        merged = window.merge(thresholds, on='trace_name', how='left')
        has_thresh = merged['global_P95'].notna() & merged['global_P5'].notna() & merged['value'].notna()
        merged['is_anomaly'] = False
        merged.loc[has_thresh, 'is_anomaly'] = (
            (merged.loc[has_thresh, 'value'] >= merged.loc[has_thresh, 'global_P95']) |
            (merged.loc[has_thresh, 'value'] <= merged.loc[has_thresh, 'global_P5'])
        )

        # Deduplicate timestamps so each timestamp counts once per trace_name
        anomalous_unique = merged[merged['is_anomaly']].drop_duplicates(subset=['trace_name','timestamp'])

        # Aggregate counts
        count_in_window = window.groupby('trace_name', observed=True)['timestamp'].count().reset_index().rename(columns={'timestamp':'count_in_window'})
        unique_anom = anomalous_unique.groupby('trace_name', observed=True)['timestamp'].agg(unique_anomaly_count='nunique', earliest_anomaly_ts='min').reset_index()

        # Combine with thresholds
        combined = thresholds.merge(count_in_window, on='trace_name', how='left').merge(unique_anom, on='trace_name', how='left')
        combined['count_in_window'] = combined['count_in_window'].fillna(0).astype(int)
        combined['unique_anomaly_count'] = combined['unique_anomaly_count'].fillna(0).astype(int)
        combined['earliest_anomaly_ts'] = combined['earliest_anomaly_ts'].where(combined['unique_anomaly_count']>0, pd.NA)
        combined['earliest_anomaly_ts_shanghai'] = combined['earliest_anomaly_ts'].apply(lambda x: ts_to_shanghai_str(int(x)) if pd.notna(x) else pd.NA)

        # Prepare final compact table and sort by unique_anomaly_count desc, limit 20
        final_traces_result = combined.loc[:, [
            'trace_name','global_P95','global_P5','count_in_window',
            'unique_anomaly_count','earliest_anomaly_ts','earliest_anomaly_ts_shanghai'
        ]].sort_values(['unique_anomaly_count','count_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)

final_traces_result

```
Out[9]:
```
Summary (plain English):

Scope: I analyzed trace rows whose trace_name contains "error_rate", "from_frontend", or "to_" within the incident window 1647862200–1647863999 (2022-03-21 19:30:00–19:59:59 Asia/Shanghai). Global P95/P5 were computed from the full series (no time filtering). Within the window a timestamp was marked anomalous per trace_name if value >= global_P95 OR value <= global_P5 (deduplicated by timestamp).

Key findings (top results, up to 20 rows):
- The highest anomaly counts are on frontend-origin traces. For example:
  - trace.from_frontend-0.duration_p95: global_P95=0.039493, global_P5=0.0, count_in_window=630, unique_anomaly_count=30, earliest anomaly=1647862200 (2022-03-21 19:30:00 CST)
  - trace.from_frontend-0.error_rate: global_P95=0.0, global_P5=0.0, count_in_window=630, unique_anomaly_count=30, earliest=1647862200
  - trace.from_frontend-2.duration_mean and duration_p95 and error_rate: each had count_in_window=628, unique_anomaly_count=30, earliest=1647862200
  - trace.from_frontend-1.* series: count_in_window ≈ 619–624, unique_anomaly_count=30, earliest=1647862200
- Many other "from_*" and "to_*" traces (checkoutservice, productcatalogservice, root, self, etc.) appear in the top 20 with unique_anomaly_count = 30 and earliest anomaly at the start of the window.

Interpretation / caveats:
- unique_anomaly_count = 30 for the top traces means every minute in the 30-minute window had at least one anomalous measurement for that trace_name (after deduplication by timestamp).
- A common pattern: global_P5 is 0.0 for many traces (especially error_rate traces) and global_P95 is also 0.0 for those error_rate metrics. When P95 == P5 == 0.0 (or when P5 equals P95 and values equal that constant), the rule (>=P95 OR <=P5) causes all equal values to be flagged — a threshold artifact that produces widespread "anomalies."
- Some non-zero P95 values exist for duration metrics (e.g., duration_p95, duration_mean), but many of those traces still show 30 anomalous timestamps, indicating the thresholds and window values also trigger the rule for every minute (possible mixture of zeros and values meeting >=P95).

Recommended next steps:
- Inspect representative raw time series for a few trace_names (e.g., a duration_p95 and an error_rate) to confirm whether the flagged hits reflect real behavioral change or threshold artifacts.
- Adjust anomaly rule to avoid equality artifacts (for example use strict >P95 / <P5, use wider percentiles like P97.5/P2.5, or consider z-score / baseline-drift detection).
- If you want, I can extract a few raw per-timestamp series for the top traces to show the actual values per minute.

The original code execution output of IPython Kernel is also provided below for reference:

trace_name  global_P95  global_P5  count_in_window  unique_anomaly_count  earliest_anomaly_ts earliest_anomaly_ts_shanghai
0            trace.from_frontend-0.duration_p95    0.039493        0.0              630                    30           1647862200      2022-03-21 19:30:00 CST
1              trace.from_frontend-0.error_rate    0.000000        0.0              630                    30           1647862200      2022-03-21 19:30:00 CST
2           trace.from_frontend-2.duration_mean    0.038239        0.0              628                    30           1647862200      2022-03-21 19:30:00 CST
3            trace.from_frontend-2.duration_p95    0.038710        0.0              628                    30           1647862200      2022-03-21 19:30:00 CST
4              trace.from_frontend-2.error_rate    0.000000        0.0              628                    30           1647862200      2022-03-21 19:30:00 CST
5               trace.from_frontend-2.row_count  143.000000        1.0              628                    30           1647862200      2022-03-21 19:30:00 CST
6           trace.from_frontend-1.duration_mean    0.038393        0.0              619                    30           1647862200      2022-03-21 19:30:00 CST
7            trace.from_frontend-1.duration_p95    0.038399        0.0              619                    30           1647862200      2022-03-21 19:30:00 CST
8              trace.from_frontend-1.error_rate    0.000000        0.0              619                    30           1647862200      2022-03-21 19:30:00 CST
9               trace.from_frontend-1.row_count  124.000000        1.0              619                    30           1647862200      2022-03-21 19:30:00 CST
10      trace.from_checkoutservice-0.error_rate    0.000000        0.0              538                    30           1647862200      2022-03-21 19:30:00 CST
11      trace.from_checkoutservice-2.error_rate    0.000000        0.0              537                    30           1647862200      2022-03-21 19:30:00 CST
12      trace.from_checkoutservice-1.error_rate    0.000000        0.0              536                    30           1647862200      2022-03-21 19:30:00 CST
13                        trace.self.error_rate    0.000000        0.0              480                    30           1647862200      2022-03-21 19:30:00 CST
14  trace.to_productcatalogservice-0.error_rate    0.000000        0.0              300                    30           1647862200      2022-03-21 19:30:00 CST
15  trace.to_productcatalogservice-1.error_rate    0.000000        0.0              300                    30           1647862200      2022-03-21 19:30:00 CST
16  trace.to_productcatalogservice-2.error_rate    0.000000        0.0              300                    30           1647862200      2022-03-21 19:30:00 CST
17                   trace.from_root.error_rate    0.000000        0.0              240                    30           1647862200      2022-03-21 19:30:00 CST
18            trace.from_frontend2-0.error_rate    0.000000        0.0              210                    30           1647862200      2022-03-21 19:30:00 CST
19     trace.from_checkoutservice2-0.error_rate    0.000000        0.0              180                    30           1647862200      2022-03-21 19:30:00 CST```
```

In [ ]:
# In[10]:

# Filter error_logs for the three frontends and produce compact, bounded outputs.
# Reuse existing `error_df` and `ts_to_shanghai_str` from prior state.

import pandas as pd

frontends = ['frontend-0','frontend-1','frontend-2']

# Ensure timestamp is integer
if 'timestamp' in error_df.columns:
    error_df['timestamp'] = error_df['timestamp'].astype('Int64')

# Filter
front_df = error_df[error_df['cmdb_id'].isin(frontends)].copy()

# 1) Top 20 distinct message prefixes (first 200 chars) with counts and earliest timestamp
if front_df.shape[0] == 0:
    message_summary_top20 = pd.DataFrame(columns=['message_text_truncated','count','earliest_timestamp_unix','earliest_timestamp_Asia/Shanghai'])
else:
    front_df['message_text_truncated'] = front_df['message'].astype(str).str.slice(0,200)
    msg_grp = (
        front_df
        .groupby('message_text_truncated', observed=True)
        .agg(count=('message','size'), earliest_timestamp_unix=('timestamp','min'))
        .reset_index()
    )
    msg_grp['earliest_timestamp_Asia/Shanghai'] = msg_grp['earliest_timestamp_unix'].apply(lambda x: ts_to_shanghai_str(int(x)) if pd.notna(x) else pd.NA)
    message_summary_top20 = msg_grp.sort_values('count', ascending=False).head(20).loc[:, ['message_text_truncated','count','earliest_timestamp_unix','earliest_timestamp_Asia/Shanghai']].reset_index(drop=True)

# 2) Overall earliest error log timestamp among these three frontends and which frontend it came from
if front_df.shape[0] == 0:
    earliest_overall = pd.DataFrame([{
        'note': 'No error_logs for the requested frontends',
        'earliest_timestamp_unix': pd.NA,
        'earliest_timestamp_Asia/Shanghai': pd.NA,
        'cmdb_id': pd.NA
    }])
else:
    idx_min = front_df['timestamp'].idxmin()
    row_min = front_df.loc[idx_min]
    earliest_overall = pd.DataFrame([{
        'earliest_timestamp_unix': int(row_min['timestamp']),
        'earliest_timestamp_Asia/Shanghai': ts_to_shanghai_str(int(row_min['timestamp'])),
        'cmdb_id': row_min['cmdb_id']
    }])

# 3) First 10 error-log rows for these frontends in chronological order
if front_df.shape[0] == 0:
    first10_rows = pd.DataFrame(columns=['timestamp','ts_Asia/Shanghai','cmdb_id','message'])
else:
    first10 = front_df.sort_values('timestamp', ascending=True).head(10).copy()
    first10['ts_Asia/Shanghai'] = first10['timestamp'].apply(lambda x: ts_to_shanghai_str(int(x)) if pd.notna(x) else pd.NA)
    # Truncate message to 400 chars for display
    first10['message'] = first10['message'].astype(str).str.slice(0,400)
    first10_rows = first10.loc[:, ['timestamp','ts_Asia/Shanghai','cmdb_id','message']].reset_index(drop=True)

# Also provide per-frontend counts and explicitly note any frontend with zero records
frontend_counts = front_df['cmdb_id'].value_counts().reindex(frontends, fill_value=0).reset_index()
frontend_counts.columns = ['cmdb_id','count']
# Create a small note DataFrame for frontends with zero records (if any)
zero_frontends = frontend_counts[frontend_counts['count']==0].copy()
if zero_frontends.shape[0] == 0:
    zero_frontend_note = pd.DataFrame([{'note':'All requested frontends have error_log records.'}])
else:
    zero_frontend_note = zero_frontends

# Final compact outputs (variables displayed)
frontend_counts, zero_frontend_note, message_summary_top20, earliest_overall, first10_rows

```
Out[10]:
```
Summary (plain English):

- Per-frontend counts (error_logs.csv):
  - frontend-0: 922 records
  - frontend-1: 464 records
  - frontend-2: 543 records
  - All three requested frontends have error log records.

- Top distinct message (truncated to first 200 chars):
  - "severity: warning, message: failed to retrieve..." — count = 1929
  - Earliest occurrence (unix) = 1647862201 → 2022-03-21 19:30:01 Asia/Shanghai

- Overall earliest error among these frontends:
  - Timestamp (unix) = 1647862201 → 2022-03-21 19:30:01 Asia/Shanghai
  - Originating frontend: frontend-2

- First 10 error-log rows for frontend-0/1/2 (chronological, message truncated to 400 chars):
  1) 1647862201 — 2022-03-21 19:30:01 CST — frontend-2 — severity: warning, message: failed to retrieve...
  2) 1647862201 — 2022-03-21 19:30:01 CST — frontend-0 — severity: warning, message: failed to retrieve...
  3) 1647862202 — 2022-03-21 19:30:02 CST — frontend-1 — severity: warning, message: failed to retrieve...
  4) 1647862203 — 2022-03-21 19:30:03 CST — frontend-0 — severity: warning, message: failed to retrieve...
  5) 1647862204 — 2022-03-21 19:30:04 CST — frontend-2 — severity: warning, message: failed to retrieve...
  6) 1647862205 — 2022-03-21 19:30:05 CST — frontend-0 — severity: warning, message: failed to retrieve...
  7) 1647862206 — 2022-03-21 19:30:06 CST — frontend-0 — severity: warning, message: failed to retrieve...
  8) 1647862206 — 2022-03-21 19:30:06 CST — frontend-2 — severity: warning, message: failed to retrieve...
  9) 1647862209 — 2022-03-21 19:30:09 CST — frontend-0 — severity: warning, message: failed to retrieve...
 10) 1647862209 — 2022-03-21 19:30:09 CST — frontend-1 — severity: warning, message: failed to retrieve...

If you want, I can:
- Expand the top-20 distinct messages list,
- Show full (untruncated) messages for specific rows,
- Or extract later error-log rows.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  count
0  frontend-0    922
1  frontend-1    464
2  frontend-2    543,                                               note
0  All requested frontends have error_log records.,                               message_text_truncated  count  earliest_timestamp_unix earliest_timestamp_Asia/Shanghai
0  severity: warning, message: failed to retrieve...   1929               1647862201          2022-03-21 19:30:01 CST,    earliest_timestamp_unix earliest_timestamp_Asia/Shanghai     cmdb_id
0               1647862201          2022-03-21 19:30:01 CST  frontend-2,     timestamp         ts_Asia/Shanghai     cmdb_id                                            message
0  1647862201  2022-03-21 19:30:01 CST  frontend-2  severity: warning, message: failed to retrieve...
1  1647862201  2022-03-21 19:30:01 CST  frontend-0  severity: warning, message: failed to retrieve...
2  1647862202  2022-03-21 19:30:02 CST  frontend-1  severity: warning, message: failed to retrieve...
3  1647862203  2022-03-21 19:30:03 CST  frontend-0  severity: warning, message: failed to retrieve...
4  1647862204  2022-03-21 19:30:04 CST  frontend-2  severity: warning, message: failed to retrieve...
5  1647862205  2022-03-21 19:30:05 CST  frontend-0  severity: warning, message: failed to retrieve...
6  1647862206  2022-03-21 19:30:06 CST  frontend-0  severity: warning, message: failed to retrieve...
7  1647862206  2022-03-21 19:30:06 CST  frontend-2  severity: warning, message: failed to retrieve...
8  1647862209  2022-03-21 19:30:09 CST  frontend-0  severity: warning, message: failed to retrieve...
9  1647862209  2022-03-21 19:30:09 CST  frontend-1  severity: warning, message: failed to retrieve...)```
```

In [ ]:
# In[11]:

# Reuse existing variables (error_df, ts_to_shanghai_str) from kernel state.
import pandas as pd

frontends = ['frontend-0','frontend-1','frontend-2']

# Ensure timestamp column is integer-like
if 'timestamp' in error_df.columns:
    error_df['timestamp'] = error_df['timestamp'].astype('Int64')

# Filter for requested frontends
front_df = error_df[error_df['cmdb_id'].isin(frontends)].copy()

# Per-frontend counts and note any frontend with zero records
frontend_counts = front_df['cmdb_id'].value_counts().reindex(frontends, fill_value=0).reset_index()
frontend_counts.columns = ['cmdb_id','count']
zero_frontends = frontend_counts[frontend_counts['count']==0].copy()

# 1) Top 20 distinct full message texts (use full message for grouping, truncate display to 1000 chars)
if front_df.shape[0] == 0:
    message_summary_top20 = pd.DataFrame(columns=['message_text_truncated_to_1000_chars','count','earliest_timestamp_unix','earliest_timestamp_Asia/Shanghai'])
else:
    # Ensure message is string
    front_df['message_full'] = front_df['message'].astype(str)
    msg_grp = (
        front_df
        .groupby('message_full', observed=True)
        .agg(count=('message_full','size'), earliest_timestamp_unix=('timestamp','min'))
        .reset_index()
        .sort_values(['count','earliest_timestamp_unix'], ascending=[False, True])
        .head(20)
    )
    msg_grp['message_text_truncated_to_1000_chars'] = msg_grp['message_full'].str.slice(0,1000)
    msg_grp['earliest_timestamp_Asia/Shanghai'] = msg_grp['earliest_timestamp_unix'].apply(lambda x: ts_to_shanghai_str(int(x)) if pd.notna(x) else pd.NA)
    message_summary_top20 = msg_grp.loc[:, ['message_text_truncated_to_1000_chars','count','earliest_timestamp_unix','earliest_timestamp_Asia/Shanghai']].reset_index(drop=True)

# 2) Overall earliest error-log row among the three frontends
if front_df.shape[0] == 0:
    earliest_overall = pd.DataFrame([{
        'note': 'No error_logs for the requested frontends',
        'earliest_timestamp_unix': pd.NA,
        'earliest_timestamp_Asia/Shanghai': pd.NA,
        'cmdb_id': pd.NA,
        'message_truncated_to_1000_chars': pd.NA
    }])
else:
    idx_min = front_df['timestamp'].idxmin()
    row_min = front_df.loc[idx_min]
    earliest_overall = pd.DataFrame([{
        'earliest_timestamp_unix': int(row_min['timestamp']),
        'earliest_timestamp_Asia/Shanghai': ts_to_shanghai_str(int(row_min['timestamp'])),
        'cmdb_id': row_min['cmdb_id'],
        'message_truncated_to_1000_chars': str(row_min['message'])[:1000]
    }])

# 3) First 20 error-log rows for these frontends in chronological order
if front_df.shape[0] == 0:
    first20_rows = pd.DataFrame(columns=['timestamp','ts_Asia/Shanghai','cmdb_id','message'])
else:
    first20 = front_df.sort_values('timestamp', ascending=True).head(20).copy()
    first20['ts_Asia/Shanghai'] = first20['timestamp'].apply(lambda x: ts_to_shanghai_str(int(x)) if pd.notna(x) else pd.NA)
    first20['message'] = first20['message'].astype(str).str.slice(0,1000)
    first20_rows = first20.loc[:, ['timestamp','ts_Asia/Shanghai','cmdb_id','message']].reset_index(drop=True)

# Prepare explicit note if any frontend has zero records
if zero_frontends.shape[0] == 0:
    zero_frontend_note = pd.DataFrame([{'note':'All requested frontends have error_log records.'}])
else:
    zero_frontend_note = zero_frontends

# Final compact outputs (variables to display)
frontend_counts, zero_frontend_note, message_summary_top20, earliest_overall, first20_rows

```
Out[11]:
```
```python
# Build a concise plain-English summary using prior results (variables from previous execution).
# Output a single summary string variable.

# Use existing variables: frontend_counts, message_summary_top20, earliest_overall, first20_rows
# If any are not present in the kernel, handle gracefully.
_exists = lambda v: v in globals()

if not _exists('frontend_counts'):
    summary = "No prior frontend error-log results available in kernel state."
else:
    # Prepare counts
    counts_list = ", ".join([f"{row['cmdb_id']}: {int(row['count'])}" for _, row in frontend_counts.iterrows()])
    # Top message (safe access)
    if _exists('message_summary_top20') and not message_summary_top20.empty:
        top_msg_row = message_summary_top20.iloc[0]
        top_msg = top_msg_row['message_text_truncated_to_1000_chars']
        top_count = int(top_msg_row['count'])
        top_earliest_unix = int(top_msg_row['earliest_timestamp_unix'])
        top_earliest_sh = top_msg_row['earliest_timestamp_Asia/Shanghai']
    else:
        top_msg = "(no messages)"
        top_count = 0
        top_earliest_unix = None
        top_earliest_sh = None

    # Earliest overall
    if _exists('earliest_overall') and 'earliest_timestamp_unix' in earliest_overall.columns and pd.notna(earliest_overall.loc[0,'earliest_timestamp_unix']):
        eo_unix = int(earliest_overall.loc[0,'earliest_timestamp_unix'])
        eo_sh = earliest_overall.loc[0,'earliest_timestamp_Asia/Shanghai']
        eo_cmdb = earliest_overall.loc[0,'cmdb_id']
    else:
        eo_unix = None
        eo_sh = None
        eo_cmdb = None

    # First 20 rows sample
    sample_lines = []
    if _exists('first20_rows') and not first20_rows.empty:
        for _, r in first20_rows.iterrows():
            ts = int(r['timestamp'])
            sh = r['ts_Asia/Shanghai']
            cm = r['cmdb_id']
            msg = str(r['message'])
            sample_lines.append(f"{ts} | {sh} | {cm} | {msg}")
    else:
        sample_lines.append("(no rows)")

    # Compose summary
    summary = (
        "Error-log diagnostics for frontends (frontend-0, frontend-1, frontend-2):\n\n"
        f"1) Per-frontend record counts: {counts_list}.\n"
        "   (All three frontends have error_log records.)\n\n"
        f"2) Top distinct message (full text displayed up to 1000 chars):\n"
        f"   \"{top_msg}\" — count = {top_count}, earliest = {top_earliest_unix} ({top_earliest_sh}).\n\n"
        f"3) Overall earliest error among these frontends: {eo_unix} ({eo_sh}), originating from {eo_cmdb}.\n\n"
        "4) First 20 chronological error-log rows (timestamp | Asia/Shanghai | cmdb_id | message):\n"
        + "\n".join(sample_lines)
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  count
0  frontend-0    922
1  frontend-1    464
2  frontend-2    543,                                               note
0  All requested frontends have error_log records.,                 message_text_truncated_to_1000_chars  count  earliest_timestamp_unix earliest_timestamp_Asia/Shanghai
0  severity: warning, message: failed to retrieve...   1929               1647862201          2022-03-21 19:30:01 CST,    earliest_timestamp_unix earliest_timestamp_Asia/Shanghai     cmdb_id                    message_truncated_to_1000_chars
0               1647862201          2022-03-21 19:30:01 CST  frontend-2  severity: warning, message: failed to retrieve...,      timestamp         ts_Asia/Shanghai     cmdb_id                                            message
0   1647862201  2022-03-21 19:30:01 CST  frontend-2  severity: warning, message: failed to retrieve...
1   1647862201  2022-03-21 19:30:01 CST  frontend-0  severity: warning, message: failed to retrieve...
2   1647862202  2022-03-21 19:30:02 CST  frontend-1  severity: warning, message: failed to retrieve...
3   1647862203  2022-03-21 19:30:03 CST  frontend-0  severity: warning, message: failed to retrieve...
4   1647862204  2022-03-21 19:30:04 CST  frontend-2  severity: warning, message: failed to retrieve...
5   1647862205  2022-03-21 19:30:05 CST  frontend-0  severity: warning, message: failed to retrieve...
6   1647862206  2022-03-21 19:30:06 CST  frontend-0  severity: warning, message: failed to retrieve...
7   1647862206  2022-03-21 19:30:06 CST  frontend-2  severity: warning, message: failed to retrieve...
8   1647862209  2022-03-21 19:30:09 CST  frontend-0  severity: warning, message: failed to retrieve...
9   1647862209  2022-03-21 19:30:09 CST  frontend-1  severity: warning, message: failed to retrieve...
10  1647862212  2022-03-21 19:30:12 CST  frontend-1  severity: warning, message: failed to retrieve...
11  1647862212  2022-03-21 19:30:12 CST  frontend-0  severity: warning, message: failed to retrieve...
12  1647862212  2022-03-21 19:30:12 CST  frontend-2  severity: warning, message: failed to retrieve...
13  1647862213  2022-03-21 19:30:13 CST  frontend-0  severity: warning, message: failed to retrieve...
14  1647862214  2022-03-21 19:30:14 CST  frontend-2  severity: warning, message: failed to retrieve...
15  1647862214  2022-03-21 19:30:14 CST  frontend-0  severity: warning, message: failed to retrieve...
16  1647862215  2022-03-21 19:30:15 CST  frontend-1  severity: warning, message: failed to retrieve...
17  1647862218  2022-03-21 19:30:18 CST  frontend-2  severity: warning, message: failed to retrieve...
18  1647862218  2022-03-21 19:30:18 CST  frontend-0  severity: warning, message: failed to retrieve...
19  1647862220  2022-03-21 19:30:20 CST  frontend-1  severity: warning, message: failed to retrieve...)```
```

In [ ]:
# In[12]:

# Reuse existing kernel variables: error_df, ts_to_shanghai_str, etc.
import pandas as pd

frontends = ['frontend-0','frontend-1','frontend-2']

# Ensure timestamp is integer and message is string
error_df['timestamp'] = error_df['timestamp'].astype('Int64')
error_df['message'] = error_df['message'].astype(str)

# Filter for requested frontends
front_df = error_df[error_df['cmdb_id'].isin(frontends)].copy()

# Note if any frontend has zero records
frontend_counts = front_df['cmdb_id'].value_counts().reindex(frontends, fill_value=0).reset_index()
frontend_counts.columns = ['cmdb_id','count']
zero_frontends = frontend_counts[frontend_counts['count']==0].copy()

# 1) Top 20 distinct full message texts (display each up to 1000 chars) with count and earliest timestamp
if front_df.shape[0] == 0:
    message_summary_top20 = pd.DataFrame(columns=[
        'message_text_truncated_to_1000_chars','count','earliest_timestamp_unix','earliest_timestamp_Asia/Shanghai'
    ])
else:
    grp = (
        front_df
        .groupby('message', observed=True)
        .agg(count=('message','size'), earliest_timestamp_unix=('timestamp','min'))
        .reset_index()
        .sort_values(['count','earliest_timestamp_unix'], ascending=[False, True])
        .head(20)
    )
    grp['message_text_truncated_to_1000_chars'] = grp['message'].str.slice(0,1000)
    grp['earliest_timestamp_Asia/Shanghai'] = grp['earliest_timestamp_unix'].apply(lambda x: ts_to_shanghai_str(int(x)) if pd.notna(x) else pd.NA)
    message_summary_top20 = grp.loc[:, ['message_text_truncated_to_1000_chars','count','earliest_timestamp_unix','earliest_timestamp_Asia/Shanghai']].reset_index(drop=True)

# 2) Keyword counts (case-insensitive). Messages may match multiple keywords.
keywords = [
    'timeout','timed out','connection refused','failed to retrieve','5','50','500','502','503',
    'dns','refused','connection reset','upstream','target','product','checkout','catalog','cart','shipping'
]
# Deduplicate keywords (there was a duplicate 'timeout' in the user list)
keywords = list(dict.fromkeys([k.lower() for k in keywords]))

if front_df.shape[0] == 0:
    keyword_counts_df = pd.DataFrame(columns=['keyword','count'])
else:
    rows = []
    # Use message field (each row) for counting
    for kw in keywords:
        # case-insensitive substring match; na=False to skip NaN
        match_count = front_df['message'].str.contains(kw, case=False, na=False).sum()
        rows.append({'keyword': kw, 'count': int(match_count)})
    keyword_counts_df = pd.DataFrame(rows).sort_values('count', ascending=False).reset_index(drop=True).head(20)

# 3) Single earliest error-log row among these frontends (with message truncated to 1000 chars)
if front_df.shape[0] == 0:
    earliest_row_df = pd.DataFrame([{
        'note':'No error_logs for requested frontends',
        'timestamp': pd.NA,
        'ts_Asia/Shanghai': pd.NA,
        'cmdb_id': pd.NA,
        'message_truncated_to_1000_chars': pd.NA
    }])
else:
    idx = int(front_df['timestamp'].idxmin())
    row = front_df.loc[idx]
    earliest_row_df = pd.DataFrame([{
        'timestamp': int(row['timestamp']),
        'ts_Asia/Shanghai': ts_to_shanghai_str(int(row['timestamp'])),
        'cmdb_id': row['cmdb_id'],
        'message_truncated_to_1000_chars': str(row['message'])[:1000]
    }])

# Final compact outputs (display variables)
frontend_counts, zero_frontends, message_summary_top20, keyword_counts_df, earliest_row_df

```
Out[12]:
```
Summary (plain English):

- Records present:
  - frontend-0: 922 error-log records
  - frontend-1: 464 error-log records
  - frontend-2: 543 error-log records
  - All three frontends have error-log entries.

1) Top distinct messages (up to 20)
- The dominant message (full text shown up to 1000 chars) is:
  "severity: warning, message: failed to retrieve..." — count = 1929
  - Earliest occurrence: 1647862201 → 2022-03-21 19:30:01 Asia/Shanghai
- No other distinct messages reached comparable counts in the top-20 (the dataset is dominated by this message).

2) Keyword counts (case-insensitive) in these frontend messages
- "failed to retrieve": 1929 matches
- All other keywords checked (timed out / timeout / connection refused / 5 / 50 / 500 / 502 / 503 / dns / refused / connection reset / upstream / target / product / checkout / catalog / cart / shipping) returned 0 matches in this filtered set.

3) Earliest error-log row among the three frontends
- Timestamp (unix): 1647862201
- Asia/Shanghai: 2022-03-21 19:30:01
- cmdb_id: frontend-2
- Message (truncated to 1000 chars): "severity: warning, message: failed to retrieve..."

Interpretation / next steps:
- The frontends' logs are overwhelmingly reporting "failed to retrieve", suggesting a common downstream retrieval failure (likely an upstream service or resource the frontends call).
- No explicit HTTP error codes or textual keywords like "timeout", "500", "checkout", "product", etc., were found in these messages — so the logs do not directly show target service names or HTTP codes.
- Recommended actions:
  - Inspect the full (untruncated) messages and any structured fields included with them to find the target/URL or error context.
  - Correlate with traces/metrics (e.g., traces to product/checkout services, and metrics on shippingservice2-0/node-5) around 19:30 to identify the downstream target.
  - Check network/DNS and upstream service health for endpoints the frontends call.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  count
0  frontend-0    922
1  frontend-1    464
2  frontend-2    543, Empty DataFrame
Columns: [cmdb_id, count]
Index: [],                 message_text_truncated_to_1000_chars  count  earliest_timestamp_unix earliest_timestamp_Asia/Shanghai
0  severity: warning, message: failed to retrieve...   1929               1647862201          2022-03-21 19:30:01 CST,                keyword  count
0   failed to retrieve   1929
1            timed out      0
2              timeout      0
3   connection refused      0
4                    5      0
5                   50      0
6                  500      0
7                  502      0
8                  503      0
9                  dns      0
10             refused      0
11    connection reset      0
12            upstream      0
13              target      0
14             product      0
15            checkout      0
16             catalog      0
17                cart      0
18            shipping      0,     timestamp         ts_Asia/Shanghai     cmdb_id                    message_truncated_to_1000_chars
0  1647862201  2022-03-21 19:30:01 CST  frontend-2  severity: warning, message: failed to retrieve...)```
```